# Land-Cover Classification — Spatial Cross-Validation (Kerkini)

**Research source code for:** Gkologkinas et al. (2026), *Remote Sensing*, 18, 2463.  
DOI: https://doi.org/10.3390/rs18152463

This notebook contains implementations of **CNN, ViT, U-Net++ patch classifier, and a ConvNeXt-style patch classifier** for multispectral Sentinel-2 land-cover classification, with geographically separated outer folds and optional spatial hyperparameter tuning.

**Important notes before running**

- The uploaded execution had **ConvNeXt enabled only**, while CNN, ViT, and U-Net++ were disabled through architecture flags. Their implementations are present, but their results are **not** reproduced in this copy.
- Requires the local GeoTIFF stack, reference raster, and (for continuation runs) stored split-cache files; these inputs are **not distributed here**. Update the `stack_path`, `ref_path`, and `split_cache_dir` values in the configuration.
- The code targets Google Colab, TensorFlow, and a GPU. Running it may consume significant GPU time.
- **Not a complete reproduction of the published paper**: in particular, this notebook does not include the independent Lake Doirani transfer assessment.
- Code is preserved from the uploaded notebook, organized into sections, and outputs were cleared to avoid distributing oversized training logs. It **has not been rerun or functionally validated after repackaging**.

See [README.md](README.md) for run instructions and the limited scope of the recorded metrics.


## 1. Install dependencies (Google Colab)


In [ ]:
!pip install -q rasterio scikit-learn tensorflow


## 2. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Spatial split design, input rasters and split-cache utilities


In [ ]:
# ============================================================
# FULL MODULAR PIPELINE: OPTION-B SPLIT + OPTIONAL HPO + FINAL EVALUATION
# Architectures: CNN, ViT, U-Net++, ConvNeXt-style patch classifier.
#
# Main guarantees:
# 1) Build/load ONE reviewer-safe Option-B outer split at the beginning.
# 2) Reuse exactly the SAME outer folds for every architecture.
# 3) Optionally run global spatial HPO on leakage-safe HPO subfolds.
# 4) Optionally skip HPO/final training if results already exist.
# 5) Train/evaluate models one by one, with separate outputs per architecture.
#
# This is intentionally one-file/one-cell friendly for Colab.
# ============================================================

import os
import json
import time
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

import gc
import traceback

try:
    import psutil
except Exception:
    psutil = None

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    f1_score,
    cohen_kappa_score,
)

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.utils import Sequence

# Optional Colab Drive mount.
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
except Exception:
    pass

try:
    from IPython.display import display, Image
except Exception:
    display = None
    Image = None

# =========================
# USER PATHS
# =========================
stack_path = "/content/drive/MyDrive/GEE_Exports/Kerkini_S2_Seasonal_Stack_2021.tif"
ref_path   = "/content/drive/MyDrive/GEE_Exports/Kerkini_WorldCover6_reference.tif"

# Cache path used later by fixed-evaluation scripts.
split_cache_dir = "/content/drive/MyDrive/GEE_Exports/fixed_spatial_splits"
SPLIT_CACHE_NAME = "kerkini_optionB_multiblock_outer4_v1"

SPLIT_MASKS_PATH = os.path.join(split_cache_dir, f"{SPLIT_CACHE_NAME}_masks.npz")
SPLIT_META_PATH  = os.path.join(split_cache_dir, f"{SPLIT_CACHE_NAME}_metadata.json")

# Local/run outputs: audit tables and figures.
out_dir = "/content/split_builder_optionB_multiblock_outer4_v1"
fig_dir = os.path.join(out_dir, "figures")
os.makedirs(split_cache_dir, exist_ok=True)
os.makedirs(out_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

# Protect existing split cache by default.
# Set True ONLY when you intentionally want to regenerate and overwrite the split.
OVERWRITE_EXISTING_SPLIT = False
SHOW_SPLIT_IMAGES = True

# Optional extras.
EXPORT_SPLIT_CLASS_COUNTS = True
EXPORT_MASK_GEOTIFFS = False              # Extra convenience; not needed by model scripts.
EXPORT_NORMALIZATION_STATS = False        # Split-only script does not need stack pixels. Set True if you want per-fold JSON stats.

# =========================
# STRICT SPLIT CONFIG
# =========================
GLOBAL_SEED = 42

valid_classes = np.array([1, 2, 3, 4, 5], dtype=np.uint8)
CLASS_NAMES = {
    1: "Water",
    2: "Cropland",
    3: "Forest",
    4: "LowNatural",
    5: "BuiltUp",
}

# Geometry inherited from the multi-architecture code.
# Important: this is based on the max patch in the original search space: [15,19,23,27,31].
MAX_PATCH_SIZE = 31
MAX_R = MAX_PATCH_SIZE // 2
GAP_PIXELS = 2 * MAX_R

# Reviewer-safe split requirement: all 5 classes must exist in train / validation / test.
MIN_CLASSES_REQUIRED = 5
MIN_SPLIT_TRAIN_PIXELS_FRAC = 0.45

# Minimum class coverage inside final split masks.
MIN_VAL_PER_CLASS = 150
MIN_TRAIN_PER_CLASS = 500
STRICT_CLASS_COVERAGE = True

# Random class-aware outer test split settings.
N_RANDOM_OUTER_FOLDS = 4
MIN_TEST_PER_CLASS = 150
N_RANDOM_CANDIDATES = 60000
CANDIDATE_KEEP_TOP = 15000
COMBINATION_TRIALS = 250000
TEST_AREA_FRAC_RANGE = (0.010, 0.050)
ASPECT_RATIO_RANGE = (0.55, 2.20)
TARGET_RARE_CLASS = 5
TARGETED_RARE_CLASS_FRACTION = 0.60
MAX_TEST_RECT_OVERLAP_FRAC = 0.0

# Option B: representative multi-block outer validation.
MULTIBLOCK_OUTER_VAL = True
FINAL_VAL_AREA_FRAC = 0.10
FINAL_VAL_H_FRAC = 0.32

OUTER_VAL_N_BLOCKS = 3
OUTER_VAL_TOTAL_AREA_FRAC = FINAL_VAL_AREA_FRAC
OUTER_VAL_BLOCK_AREA_FRAC = OUTER_VAL_TOTAL_AREA_FRAC / OUTER_VAL_N_BLOCKS
OUTER_VAL_BLOCK_H_FRAC = max(0.10, min(0.22, FINAL_VAL_H_FRAC / np.sqrt(OUTER_VAL_N_BLOCKS)))

OUTER_VAL_RANDOM_CENTERS = 150
OUTER_VAL_GRID_N = 7
OUTER_VAL_TOP_CANDIDATES = 80
OUTER_VAL_COMBO_TRIALS = 2500

# Distribution constraints for validation UNION.
OUTER_VAL_MAX_SINGLE_CLASS_FRAC = 0.65
OUTER_VAL_MAX_TVD_FROM_DEV = 0.40
OUTER_VAL_MIN_TOTAL_PIXELS = 1500

OUTER_VAL_MIN_PER_CLASS = MIN_VAL_PER_CLASS
OUTER_TRAIN_MIN_PER_CLASS = MIN_TRAIN_PER_CLASS

# Kept for metadata continuity. With RANDOM_OUTER names these defaults are not usually used.
SPLIT_SEARCH_SEEDS = {
    "OUTER_01_TEST_TL_FINAL": 2101,
    "OUTER_02_TEST_TR_FINAL": 2102,
    "OUTER_03_TEST_C_FINAL": 2103,
    "OUTER_04_TEST_BR_FINAL": 2104,
    "OUTER_05_TEST_BL_FINAL": 2105,
}

os.environ["PYTHONHASHSEED"] = str(GLOBAL_SEED)
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

# =========================
# SMALL HELPERS
# =========================
def stage(msg):
    print("\n" + "=" * 90)
    print(msg)
    print("=" * 90)


def to_native(x):
    if isinstance(x, dict):
        return {str(k): to_native(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [to_native(v) for v in x]
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.floating):
        return float(x)
    if isinstance(x, np.ndarray):
        return x.tolist()
    return x


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_native(obj), f, indent=2)


def maybe_show_image(path, title=None):
    if not SHOW_SPLIT_IMAGES:
        return
    if display is None or Image is None:
        return
    if os.path.exists(path):
        if title:
            print(title)
        try:
            display(Image(filename=path))
        except Exception:
            pass


# =========================
# LOAD RASTER GRID + ALIGN REFERENCE
# =========================
stage("1/7 LOAD STACK GRID")
with rasterio.open(stack_path) as src:
    stack_profile = src.profile.copy()
    stack_transform = src.transform
    stack_crs = src.crs
    rows, cols = src.height, src.width
    band_count = src.count
    band_names = list(src.descriptions)

if band_names is None or len(band_names) == 0 or all(b is None for b in band_names):
    band_names = [f"band_{i+1}" for i in range(band_count)]
else:
    band_names = [b if b is not None else f"band_{i+1}" for i, b in enumerate(band_names)]

print("Raster grid:", rows, "rows x", cols, "cols")
print("Bands:", band_count)

stage("2/7 LOAD + ALIGN REFERENCE")
with rasterio.open(ref_path) as ref_src:
    ref_aligned = np.zeros((rows, cols), dtype=np.uint8)
    reproject(
        source=rasterio.band(ref_src, 1),
        destination=ref_aligned,
        src_transform=ref_src.transform,
        src_crs=ref_src.crs,
        dst_transform=stack_transform,
        dst_crs=stack_crs,
        resampling=Resampling.nearest,
    )

ref_aligned = ref_aligned.copy()
ref_aligned[ref_aligned == 6] = 0
label_mask = np.isin(ref_aligned, valid_classes)

print("Reference aligned shape:", ref_aligned.shape)
print("Valid labeled pixels:", int(label_mask.sum()))

# =========================
# GEOMETRY + SPLIT HELPERS
# =========================
def rect_to_mask(rect, rows, cols):
    r0, r1, c0, c1 = rect
    m = np.zeros((rows, cols), dtype=bool)
    m[r0:r1, c0:c1] = True
    return m


def dilate_rect(rect, pad, rows, cols):
    r0, r1, c0, c1 = rect
    return (
        max(0, r0 - pad),
        min(rows, r1 + pad),
        max(0, c0 - pad),
        min(cols, c1 + pad),
    )


def get_coords(mask):
    return np.argwhere(mask).astype(np.int32)


def mask_bbox(mask):
    coords = np.argwhere(mask)
    if coords.size == 0:
        return None
    return (
        int(coords[:, 0].min()),
        int(coords[:, 0].max()) + 1,
        int(coords[:, 1].min()),
        int(coords[:, 1].max()) + 1,
    )


def build_center_rect(center_r, center_c, h, w, rows, cols):
    h = int(max(1, min(int(h), rows)))
    w = int(max(1, min(int(w), cols)))
    r0 = int(round(center_r - h / 2))
    c0 = int(round(center_c - w / 2))
    r0 = max(0, min(r0, rows - h))
    c0 = max(0, min(c0, cols - w))
    return (r0, r0 + h, c0, c0 + w)


def dilate_one_pixel(mask):
    p = np.pad(mask.astype(bool), 1, mode="constant", constant_values=False)
    neighbors = [
        p[0:-2, 0:-2], p[0:-2, 1:-1], p[0:-2, 2:],
        p[1:-1, 0:-2], p[1:-1, 1:-1], p[1:-1, 2:],
        p[2:, 0:-2], p[2:, 1:-1], p[2:, 2:],
    ]
    out = np.zeros_like(mask, dtype=bool)
    for n in neighbors:
        out |= n
    return out


def masks_touch(mask_a, mask_b):
    if (mask_a & mask_b).any():
        return True
    return bool(
        (dilate_one_pixel(mask_a) & mask_b).any()
        or (dilate_one_pixel(mask_b) & mask_a).any()
    )


def class_counts_in_mask(mask, labels):
    coords = np.argwhere(mask)
    out = {int(c): 0 for c in valid_classes}
    if len(coords) == 0:
        return out
    vals = labels[coords[:, 0], coords[:, 1]]
    for c in valid_classes:
        out[int(c)] = int((vals == int(c)).sum())
    return out


def count_present_classes(counts_dict):
    return int(sum(int(v) > 0 for v in counts_dict.values()))


def build_common_valid_centers(rows, cols, label_mask, max_patch_size, gap_pixels):
    r = max_patch_size // 2
    center_margin = r + gap_pixels
    valid = np.zeros((rows, cols), dtype=bool)
    valid[center_margin:rows-center_margin, center_margin:cols-center_margin] = True
    valid &= label_mask
    return valid



def position_target_rc(position, rows, cols):
    frac = {
        "top_left": (0.22, 0.22),
        "top_right": (0.22, 0.78),
        "bottom_left": (0.78, 0.22),
        "bottom_right": (0.78, 0.78),
        "center": (0.50, 0.50),
    }[position]
    return int(round(frac[0] * (rows - 1))), int(round(frac[1] * (cols - 1)))


def make_quadrant_region_mask(position, base_mask, rows, cols, expansion=0):
    """Return a broad search region for spatial HPO validation placement."""
    r_mid = rows // 2
    c_mid = cols // 2
    exp = int(max(0, expansion))

    if position == "top_left":
        rect = (0, min(rows, r_mid + exp), 0, min(cols, c_mid + exp))
    elif position == "top_right":
        rect = (0, min(rows, r_mid + exp), max(0, c_mid - exp), cols)
    elif position == "bottom_left":
        rect = (max(0, r_mid - exp), rows, 0, min(cols, c_mid + exp))
    elif position == "bottom_right":
        rect = (max(0, r_mid - exp), rows, max(0, c_mid - exp), cols)
    elif position == "center":
        h = max(1, rows // 2 + 2 * exp)
        w = max(1, cols // 2 + 2 * exp)
        rect = build_center_rect(rows // 2, cols // 2, h, w, rows, cols)
    else:
        raise ValueError(f"Unknown position: {position}")

    return rect_to_mask(rect, rows, cols) & base_mask


def search_validation_block(
    candidate_mask, labels, position, area_frac, h_frac, split_name,
    extra_test_mask=None, extra_test_buffer_mask=None, base_mask_for_train_frac=None,
    min_val_pixels=3000, min_val_per_class=None, min_train_per_class=None,
    scales=(1.00, 0.90, 0.80, 0.70, 0.60), n_grid=7, seed=None,
    n_random_centers=48, min_train_pixels_frac=MIN_SPLIT_TRAIN_PIXELS_FRAC,
):
    """
    Single-block validation search used only for HPO subfolds.
    The final outer validation remains Option-B multi-block.
    """
    if min_val_per_class is None:
        min_val_per_class = MIN_VAL_PER_CLASS
    if min_train_per_class is None:
        min_train_per_class = MIN_TRAIN_PER_CLASS

    bbox = mask_bbox(candidate_mask)
    if bbox is None:
        raise RuntimeError(f"{split_name}: empty candidate mask.")

    rmin, rmax, cmin, cmax = bbox[0], bbox[1] - 1, bbox[2], bbox[3] - 1
    target_r, target_c = position_target_rc(position, rows, cols)

    grid_rs = np.linspace(rmin, rmax, n_grid).astype(int)
    grid_cs = np.linspace(cmin, cmax, n_grid).astype(int)
    centers = [(target_r, target_c)] + [(int(rr), int(cc)) for rr in grid_rs for cc in grid_cs]

    if seed is not None and n_random_centers and n_random_centers > 0:
        rng_centers = np.random.default_rng(int(seed))
        random_rs = rng_centers.integers(rmin, rmax + 1, size=int(n_random_centers))
        random_cs = rng_centers.integers(cmin, cmax + 1, size=int(n_random_centers))
        centers += [(int(rr), int(cc)) for rr, cc in zip(random_rs, random_cs)]

    centers = list(dict.fromkeys(centers))
    denom_train = int(base_mask_for_train_frac.sum()) if base_mask_for_train_frac is not None else int(candidate_mask.sum())

    cand_ii = make_integral(candidate_mask)
    class_iis = {int(cls): make_integral(candidate_mask & (labels == int(cls))) for cls in valid_classes}
    candidate_total = rect_sum_from_integral(cand_ii, (0, rows, 0, cols))
    candidate_class_totals = {
        int(cls): rect_sum_from_integral(class_iis[int(cls)], (0, rows, 0, cols))
        for cls in valid_classes
    }

    best = None
    best_score = -1e30

    for scale in scales:
        area = float(area_frac) * float(scale)
        w_frac = area / max(1e-9, float(h_frac))
        val_h = min(max(1, int(round(rows * float(h_frac)))), rows)
        val_w = min(max(1, int(round(cols * w_frac))), cols)

        for rr, cc in centers:
            rect = build_center_rect(rr, cc, val_h, val_w, rows, cols)
            val_n = rect_sum_from_integral(cand_ii, rect)
            if val_n < int(min_val_pixels):
                continue

            val_buffer_rect = dilate_rect(rect, GAP_PIXELS, rows, cols)
            buffer_plus_val_n = rect_sum_from_integral(cand_ii, val_buffer_rect)
            train_n = candidate_total - buffer_plus_val_n
            if train_n < int(float(min_train_pixels_frac) * denom_train):
                continue

            val_counts = {}
            train_counts = {}
            for cls in valid_classes:
                cls = int(cls)
                v = rect_sum_from_integral(class_iis[cls], rect)
                bv = rect_sum_from_integral(class_iis[cls], val_buffer_rect)
                val_counts[cls] = int(v)
                train_counts[cls] = int(candidate_class_totals[cls] - bv)

            if STRICT_CLASS_COVERAGE:
                if count_present_classes(val_counts) < len(valid_classes):
                    continue
                if count_present_classes(train_counts) < len(valid_classes):
                    continue
            else:
                if count_present_classes(val_counts) < MIN_CLASSES_REQUIRED:
                    continue
                if count_present_classes(train_counts) < MIN_CLASSES_REQUIRED:
                    continue

            if min(val_counts.values()) < int(min_val_per_class):
                continue
            if min(train_counts.values()) < int(min_train_per_class):
                continue

            counts_arr = np.array([val_counts[int(c)] for c in valid_classes], dtype=np.float64)
            min_count = float(counts_arr.min())
            balance_penalty = float(counts_arr.std())
            center_r = 0.5 * (rect[0] + rect[1])
            center_c = 0.5 * (rect[2] + rect[3])
            dist = ((center_r - target_r) / max(1, rows)) ** 2 + ((center_c - target_c) / max(1, cols)) ** 2
            score = (
                1_000_000.0 * count_present_classes(val_counts)
                + 100.0 * min_count
                + 0.02 * val_n
                - 0.10 * balance_penalty
                - 500_000.0 * dist
            )

            if score > best_score:
                best_score = score
                best = {
                    "rect": rect,
                    "val_buffer_rect": val_buffer_rect,
                    "val_counts": val_counts,
                    "train_counts": train_counts,
                    "scale": float(scale),
                    "target_position": position,
                    "selected_center_rc": [int(round(center_r)), int(round(center_c))],
                    "target_center_rc": [int(target_r), int(target_c)],
                    "score": float(score),
                }

    if best is None:
        raise RuntimeError(
            f"Could not build {split_name}. Try reducing min_val_pixels/min_val_per_class/"
            f"min_train_per_class or increasing the search region."
        )

    rect = best["rect"]
    val_buffer_rect = best["val_buffer_rect"]
    val_mask = rect_to_mask(rect, rows, cols) & candidate_mask
    val_buffer_mask = rect_to_mask(val_buffer_rect, rows, cols) & candidate_mask & (~val_mask)
    train_mask = candidate_mask & (~val_mask) & (~val_buffer_mask)

    assert_split_ok(
        train_mask=train_mask,
        val_mask=val_mask,
        test_mask=extra_test_mask,
        val_buffer_mask=val_buffer_mask,
        test_buffer_mask=extra_test_buffer_mask,
        split_name=split_name,
    )

    best["train_mask"] = train_mask
    best["val_mask"] = val_mask
    best["val_buffer_mask"] = val_buffer_mask
    best["val_counts"] = class_counts_in_mask(val_mask, labels)
    best["train_counts"] = class_counts_in_mask(train_mask, labels)
    return best


def assert_split_ok(train_mask, val_mask, test_mask=None, val_buffer_mask=None, test_buffer_mask=None, split_name=""):
    assert int((train_mask & val_mask).sum()) == 0, f"{split_name}: train/val overlap"
    assert not masks_touch(train_mask, val_mask), f"{split_name}: train/val touch"

    if val_buffer_mask is not None:
        assert int((val_mask & val_buffer_mask).sum()) == 0, f"{split_name}: val/val_buffer overlap"
        assert int((train_mask & val_buffer_mask).sum()) == 0, f"{split_name}: train/val_buffer overlap"

    if test_mask is not None:
        assert int((train_mask & test_mask).sum()) == 0, f"{split_name}: train/test overlap"
        assert int((val_mask & test_mask).sum()) == 0, f"{split_name}: val/test overlap"
        assert not masks_touch(train_mask, test_mask), f"{split_name}: train/test touch"
        assert not masks_touch(val_mask, test_mask), f"{split_name}: val/test touch"

    if test_buffer_mask is not None and test_mask is not None:
        assert int((test_mask & test_buffer_mask).sum()) == 0, f"{split_name}: test/test_buffer overlap"


def rect_sum_from_integral(ii, rect):
    r0, r1, c0, c1 = rect
    return int(ii[r1, c1] - ii[r0, c1] - ii[r1, c0] + ii[r0, c0])


def make_integral(mask):
    ii = np.zeros((mask.shape[0] + 1, mask.shape[1] + 1), dtype=np.int64)
    ii[1:, 1:] = mask.astype(np.int64).cumsum(axis=0).cumsum(axis=1)
    return ii


def rect_area(rect):
    r0, r1, c0, c1 = [int(v) for v in rect]
    return max(0, r1 - r0) * max(0, c1 - c0)


def rect_intersection_area(a, b):
    ar0, ar1, ac0, ac1 = [int(v) for v in a]
    br0, br1, bc0, bc1 = [int(v) for v in b]
    h = max(0, min(ar1, br1) - max(ar0, br0))
    w = max(0, min(ac1, bc1) - max(ac0, bc0))
    return int(h * w)


def rects_compatible_no_test_overlap(a, b, max_overlap_frac=0.0):
    inter = rect_intersection_area(a, b)
    if inter == 0:
        return True
    denom = max(1, min(rect_area(a), rect_area(b)))
    return (inter / denom) <= float(max_overlap_frac)


def rects_intersect(a, b):
    ar0, ar1, ac0, ac1 = a
    br0, br1, bc0, bc1 = b
    return not (ar1 <= br0 or br1 <= ar0 or ac1 <= bc0 or bc1 <= ac0)


def rect_center(rect):
    r0, r1, c0, c1 = rect
    return 0.5 * (r0 + r1), 0.5 * (c0 + c1)


def counts_to_array(counts):
    return np.array([counts.get(int(cls), 0) for cls in valid_classes], dtype=np.float64)


def counts_props(counts):
    arr = counts_to_array(counts)
    s = float(arr.sum())
    if s <= 0:
        return np.ones(len(valid_classes), dtype=np.float64) / len(valid_classes)
    return arr / s


def total_variation_distance(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    return 0.5 * float(np.abs(p - q).sum())


# =========================
# OUTER TEST FOLD SEARCH
# =========================
def random_rect_from_center(center_r, center_c, h, w, rows, cols):
    h = int(max(2, min(h, rows)))
    w = int(max(2, min(w, cols)))
    r0 = int(round(center_r - h / 2))
    c0 = int(round(center_c - w / 2))
    r0 = max(0, min(r0, rows - h))
    c0 = max(0, min(c0, cols - w))
    return (r0, r0 + h, c0, c0 + w)


def class_counts_from_rect(rect, class_integrals):
    return {int(cls): int(rect_sum_from_integral(ii, rect)) for cls, ii in class_integrals.items()}


def generate_candidate_test_rectangles(labels, common_valid_centers, class_integrals):
    rng = np.random.default_rng(GLOBAL_SEED)
    rare_coords = np.argwhere((labels == TARGET_RARE_CLASS) & common_valid_centers)
    all_valid_coords = np.argwhere(common_valid_centers)
    raster_area = rows * cols

    candidates = []
    seen = set()

    for i in range(int(N_RANDOM_CANDIDATES)):
        use_rare_center = (len(rare_coords) > 0) and (rng.random() < TARGETED_RARE_CLASS_FRACTION)
        if use_rare_center:
            center = rare_coords[rng.integers(0, len(rare_coords))]
        else:
            center = all_valid_coords[rng.integers(0, len(all_valid_coords))]

        cr, cc = int(center[0]), int(center[1])
        amin, amax = TEST_AREA_FRAC_RANGE
        area_frac = float(np.exp(rng.uniform(np.log(amin), np.log(amax))))
        target_area = area_frac * raster_area
        aspect = float(np.exp(rng.uniform(np.log(ASPECT_RATIO_RANGE[0]), np.log(ASPECT_RATIO_RANGE[1]))))

        h = int(round(np.sqrt(target_area / aspect)))
        w = int(round(h * aspect))
        rect = random_rect_from_center(cr, cc, h, w, rows, cols)

        if rect in seen:
            continue
        seen.add(rect)

        counts = class_counts_from_rect(rect, class_integrals)
        min_count = int(min(counts.values()))
        if min_count < int(MIN_TEST_PER_CLASS):
            continue

        total = int(sum(counts.values()))
        balance_std = float(np.std(list(counts.values())))
        score = float(min_count * 1000 + total * 0.02 - balance_std * 0.05)

        r0, r1, c0, c1 = rect
        candidates.append({
            "rect": rect,
            "center_r": float((r0 + r1) / 2),
            "center_c": float((c0 + c1) / 2),
            "height": int(r1 - r0),
            "width": int(c1 - c0),
            "test_pixels": int(total),
            "counts": counts,
            "min_class_count": int(min_count),
            "score": score,
        })

    candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)
    if len(candidates) == 0:
        raise RuntimeError("No outer test candidates found. Relax MIN_TEST_PER_CLASS or TEST_AREA_FRAC_RANGE.")

    print(f"Outer test candidates kept: {min(len(candidates), int(CANDIDATE_KEEP_TOP))}/{len(candidates)}")
    return candidates[:int(CANDIDATE_KEEP_TOP)]


def outer_set_score(combo):
    centers = np.array([[c["center_r"] / rows, c["center_c"] / cols] for c in combo], dtype=float)
    dists = [
        float(np.linalg.norm(centers[i] - centers[j]))
        for i in range(len(combo))
        for j in range(i + 1, len(combo))
    ]

    min_dist = min(dists) if dists else 0.0
    mean_dist = float(np.mean(dists)) if dists else 0.0
    mean_min_class = float(np.mean([c["min_class_count"] for c in combo]))
    total_pixels = float(np.sum([c["test_pixels"] for c in combo]))

    areas = [float(c["test_pixels"]) for c in combo]
    mean_area = float(np.mean(areas)) if areas else 0.0
    cv_area = float(np.std(areas) / mean_area) if mean_area > 0 else 0.0
    max_min_ratio = float(max(areas) / max(1.0, min(areas))) if areas else 1.0
    size_penalty = (cv_area * 100000.0) + ((max_min_ratio - 1.0) * 20000.0)

    return (
        (min_dist * 200000.0)
        + (mean_dist * 50000.0)
        + (mean_min_class * 5.0)
        + (total_pixels * 0.001)
        - size_penalty
    )


def compatible_outer_combo(combo):
    for i in range(len(combo)):
        for j in range(i + 1, len(combo)):
            if not rects_compatible_no_test_overlap(combo[i]["rect"], combo[j]["rect"], MAX_TEST_RECT_OVERLAP_FRAC):
                return False
    return True


def select_best_random_outer_set(candidates):
    rng = np.random.default_rng(GLOBAL_SEED + 999)

    stage("3/7 SELECT BEST COMPATIBLE SET OF 4 RANDOM OUTER TEST FOLDS")
    print("Applying spatial NMS for geographic diversity...")

    diverse_candidates = []
    for c in candidates:
        too_similar = False
        for dc in diverse_candidates:
            if not rects_compatible_no_test_overlap(c["rect"], dc["rect"], 0.15):
                too_similar = True
                break
        if not too_similar:
            diverse_candidates.append(c)
        if len(diverse_candidates) >= 400:
            break

    print(f"From {len(candidates)} candidates, kept {len(diverse_candidates)} geographically diverse candidates.")

    n = len(diverse_candidates)
    if n < int(N_RANDOM_OUTER_FOLDS):
        raise RuntimeError("Not enough geographically separated outer test candidates.")

    best_combo = None
    best_score = -1e18

    for t in range(int(COMBINATION_TRIALS)):
        idx = rng.choice(n, size=int(N_RANDOM_OUTER_FOLDS), replace=False)
        combo = [diverse_candidates[int(k)] for k in idx]
        if not compatible_outer_combo(combo):
            continue
        sc = outer_set_score(combo)
        if sc > best_score:
            best_score = sc
            best_combo = combo

    if best_combo is None:
        print("Random search failed. Trying deterministic backtracking...")
        partial = []
        best_bt = None
        best_bt_score = -1e18

        def backtrack(start, depth):
            nonlocal best_bt, best_bt_score
            if depth == int(N_RANDOM_OUTER_FOLDS):
                sc = outer_set_score(partial)
                if sc > best_bt_score:
                    best_bt_score = sc
                    best_bt = list(partial)
                return

            for k in range(start, len(diverse_candidates)):
                cand = diverse_candidates[k]
                ok = True
                for existing in partial:
                    if not rects_compatible_no_test_overlap(cand["rect"], existing["rect"], MAX_TEST_RECT_OVERLAP_FRAC):
                        ok = False
                        break
                if not ok:
                    continue

                partial.append(cand)
                backtrack(k + 1, depth + 1)
                partial.pop()

                if best_bt is not None and depth == 0:
                    return

        backtrack(0, 0)
        best_combo = best_bt
        best_score = best_bt_score

    if best_combo is None:
        raise RuntimeError("Could not select compatible non-overlapping outer folds.")

    best_combo = sorted(best_combo, key=lambda c: (c["center_r"], c["center_c"]))

    selected = []
    for i, c in enumerate(best_combo, 1):
        fold = dict(c)
        fold["fold_name"] = f"RANDOM_OUTER_{i:02d}"
        fold["buffer_rect"] = dilate_rect(fold["rect"], GAP_PIXELS, rows, cols)
        selected.append(fold)

    print("Selected outer test folds:")
    for fold in selected:
        print(f"  {fold['fold_name']} rect={fold['rect']} counts={fold['counts']}")

    return selected


# =========================
# OPTION-B MULTI-BLOCK VALIDATION
# =========================
def make_outer_val_candidate_rectangles(dev_mask, labels, fold_name, seed=1234):
    """
    Candidate small validation rectangles inside the dev area.
    Individual blocks do NOT need all classes; the selected union must have all classes.
    """
    rng = np.random.default_rng(int(seed))
    bbox = mask_bbox(dev_mask)
    if bbox is None:
        raise RuntimeError(f"{fold_name}: empty dev_mask")

    r0, r1, c0, c1 = bbox
    rmin, rmax = int(r0), int(r1) - 1
    cmin, cmax = int(c0), int(c1) - 1

    grid_rs = np.linspace(rmin, rmax, OUTER_VAL_GRID_N).astype(int)
    grid_cs = np.linspace(cmin, cmax, OUTER_VAL_GRID_N).astype(int)
    centers = [(int(rr), int(cc)) for rr in grid_rs for cc in grid_cs]

    rand_rs = rng.integers(rmin, rmax + 1, size=OUTER_VAL_RANDOM_CENTERS)
    rand_cs = rng.integers(cmin, cmax + 1, size=OUTER_VAL_RANDOM_CENTERS)
    centers += [(int(rr), int(cc)) for rr, cc in zip(rand_rs, rand_cs)]
    centers = list(dict.fromkeys(centers))

    dev_ii = make_integral(dev_mask)
    class_iis = {int(cls): make_integral(dev_mask & (labels == int(cls))) for cls in valid_classes}

    dev_counts = class_counts_in_mask(dev_mask, labels)
    dev_props = counts_props(dev_counts)

    candidates = []
    seen_rects = set()

    area_scales = [1.20, 1.00, 0.85, 0.70]
    h_scales = [0.80, 1.00, 1.20]

    for area_scale in area_scales:
        block_area_frac = OUTER_VAL_BLOCK_AREA_FRAC * area_scale
        for h_scale in h_scales:
            h_frac = OUTER_VAL_BLOCK_H_FRAC * h_scale
            w_frac = block_area_frac / max(1e-9, h_frac)

            h = max(6, min(rows, int(round(rows * h_frac))))
            w = max(6, min(cols, int(round(cols * w_frac))))

            for rr, cc in centers:
                rect = build_center_rect(rr, cc, h, w, rows, cols)
                if rect in seen_rects:
                    continue
                seen_rects.add(rect)

                val_n = rect_sum_from_integral(dev_ii, rect)
                if val_n < max(200, OUTER_VAL_MIN_TOTAL_PIXELS // OUTER_VAL_N_BLOCKS):
                    continue

                val_counts = {}
                for cls in valid_classes:
                    cls = int(cls)
                    val_counts[cls] = int(rect_sum_from_integral(class_iis[cls], rect))

                present = count_present_classes(val_counts)
                if present < 2:
                    continue

                props = counts_props(val_counts)
                tvd = total_variation_distance(props, dev_props)
                max_frac = float(props.max())
                min_nonzero = min([v for v in val_counts.values() if v > 0] or [0])

                score = (
                    5000.0 * present
                    + 0.80 * val_n
                    + 10.0 * min_nonzero
                    - 3500.0 * tvd
                    - 2500.0 * max(0.0, max_frac - 0.65)
                )

                candidates.append({
                    "rect": rect,
                    "buffer_rect": dilate_rect(rect, GAP_PIXELS, rows, cols),
                    "val_n": int(val_n),
                    "val_counts": val_counts,
                    "present": int(present),
                    "score": float(score),
                    "center": rect_center(rect),
                    "props": props,
                })

    candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)
    candidates = candidates[:OUTER_VAL_TOP_CANDIDATES]

    if len(candidates) < OUTER_VAL_N_BLOCKS:
        raise RuntimeError(f"{fold_name}: too few validation candidates ({len(candidates)})")

    print(f"[{fold_name}] Multi-block validation candidates: {len(candidates)}")
    return candidates


def compatible_with_selected(candidate, selected):
    """Validation block + buffer must not intersect another block + buffer."""
    for other in selected:
        if rects_intersect(candidate["buffer_rect"], other["buffer_rect"]):
            return False
    return True


def build_masks_from_val_rects(dev_mask, rects):
    val_mask = np.zeros((rows, cols), dtype=bool)
    val_buffer_mask = np.zeros((rows, cols), dtype=bool)

    for rect in rects:
        one_val = rect_to_mask(rect, rows, cols) & dev_mask
        one_buf = rect_to_mask(dilate_rect(rect, GAP_PIXELS, rows, cols), rows, cols) & dev_mask & (~one_val)
        val_mask |= one_val
        val_buffer_mask |= one_buf

    val_buffer_mask &= dev_mask & (~val_mask)
    train_mask = dev_mask & (~val_mask) & (~val_buffer_mask)
    return train_mask, val_mask, val_buffer_mask


def score_selected_val_blocks(selected, dev_mask, labels):
    rects = [x["rect"] for x in selected]
    train_mask, val_mask, val_buffer_mask = build_masks_from_val_rects(dev_mask, rects)

    val_counts = class_counts_in_mask(val_mask, labels)
    train_counts = class_counts_in_mask(train_mask, labels)
    dev_counts = class_counts_in_mask(dev_mask, labels)

    val_arr = counts_to_array(val_counts)
    train_arr = counts_to_array(train_counts)
    val_total = float(val_arr.sum())
    dev_total = float(sum(dev_counts.values()))

    if val_total < OUTER_VAL_MIN_TOTAL_PIXELS:
        return None

    if count_present_classes(val_counts) < len(valid_classes):
        return None
    if count_present_classes(train_counts) < len(valid_classes):
        return None
    if float(val_arr.min()) < float(OUTER_VAL_MIN_PER_CLASS):
        return None
    if float(train_arr.min()) < float(OUTER_TRAIN_MIN_PER_CLASS):
        return None

    val_props = counts_props(val_counts)
    dev_props = counts_props(dev_counts)
    tvd = total_variation_distance(val_props, dev_props)
    max_frac = float(val_props.max())

    if max_frac > OUTER_VAL_MAX_SINGLE_CLASS_FRAC:
        return None
    if tvd > OUTER_VAL_MAX_TVD_FROM_DEV:
        return None

    desired_val_total = max(1.0, dev_total * OUTER_VAL_TOTAL_AREA_FRAC)
    area_error = abs(val_total - desired_val_total) / desired_val_total

    centers = np.array([x["center"] for x in selected], dtype=np.float64)
    if len(centers) > 1:
        dists = []
        for i in range(len(centers)):
            for j in range(i + 1, len(centers)):
                dr = (centers[i, 0] - centers[j, 0]) / max(1, rows)
                dc = (centers[i, 1] - centers[j, 1]) / max(1, cols)
                dists.append(np.sqrt(dr * dr + dc * dc))
        spatial_spread = float(np.mean(dists))
    else:
        spatial_spread = 0.0

    min_val_class = float(val_arr.min())

    score = (
        100000.0
        + 50.0 * min_val_class
        + 0.02 * val_total
        + 1500.0 * spatial_spread
        - 30000.0 * tvd
        - 12000.0 * area_error
        - 20000.0 * max(0.0, max_frac - 0.45)
    )

    return {
        "score": float(score),
        "rects": rects,
        "train_mask": train_mask,
        "val_mask": val_mask,
        "val_buffer_mask": val_buffer_mask,
        "val_counts": val_counts,
        "train_counts": train_counts,
        "dev_counts": dev_counts,
        "tvd_from_dev": float(tvd),
        "max_class_frac": float(max_frac),
        "val_total": int(val_total),
        "area_error": float(area_error),
        "spatial_spread": float(spatial_spread),
    }


def search_representative_multiblock_validation(dev_mask, labels, fold_name, seed=1234):
    """
    Select OUTER_VAL_N_BLOCKS validation blocks from dev_mask.
    Keeps outer test fixed because dev_mask already excludes test + test buffer.
    """
    rng = np.random.default_rng(int(seed))
    candidates = make_outer_val_candidate_rectangles(dev_mask, labels, fold_name, seed=seed)

    best = None

    # 1) Deterministic greedy starts from strong candidates.
    starts = candidates[:min(80, len(candidates))]
    for start in starts:
        selected = [start]
        pool = candidates.copy()
        rng.shuffle(pool)
        pool = sorted(pool, key=lambda x: x["score"], reverse=True)

        while len(selected) < OUTER_VAL_N_BLOCKS:
            added = False
            for cand in pool:
                if any(cand["rect"] == s["rect"] for s in selected):
                    continue
                if compatible_with_selected(cand, selected):
                    selected.append(cand)
                    added = True
                    break
            if not added:
                break

        if len(selected) == OUTER_VAL_N_BLOCKS:
            scored = score_selected_val_blocks(selected, dev_mask, labels)
            if scored is not None and (best is None or scored["score"] > best["score"]):
                best = scored

    # 2) Random weighted combination search.
    n = len(candidates)
    weights = np.array(
        [max(1e-6, c["score"] - candidates[-1]["score"] + 1.0) for c in candidates],
        dtype=np.float64,
    )
    weights = weights / weights.sum()

    for trial in range(int(OUTER_VAL_COMBO_TRIALS)):
        if trial % 500 == 0:
            print(f"[{fold_name}] combo trial {trial}/{OUTER_VAL_COMBO_TRIALS} | best={'yes' if best is not None else 'no'}")

        selected = []
        idxs = rng.choice(n, size=min(n, OUTER_VAL_N_BLOCKS * 8), replace=False, p=weights)

        for idx in idxs:
            cand = candidates[int(idx)]
            if compatible_with_selected(cand, selected):
                selected.append(cand)
                if len(selected) == OUTER_VAL_N_BLOCKS:
                    break

        if len(selected) != OUTER_VAL_N_BLOCKS:
            continue

        scored = score_selected_val_blocks(selected, dev_mask, labels)
        if scored is not None and (best is None or scored["score"] > best["score"]):
            best = scored

    if best is None:
        raise RuntimeError(
            f"{fold_name}: could not find representative multi-block validation. "
            f"Try relaxing OUTER_VAL_MAX_SINGLE_CLASS_FRAC or OUTER_VAL_MAX_TVD_FROM_DEV, "
            f"or set OUTER_VAL_N_BLOCKS=4."
        )

    assert_split_ok(
        train_mask=best["train_mask"],
        val_mask=best["val_mask"],
        test_mask=None,
        val_buffer_mask=best["val_buffer_mask"],
        test_buffer_mask=None,
        split_name=fold_name,
    )

    print(f"\n✅ {fold_name} representative multi-block validation")
    print(f"   rects: {best['rects']}")
    print(f"   val_counts: {best['val_counts']}")
    print(f"   train_counts: {best['train_counts']}")
    print(f"   max_class_frac={best['max_class_frac']:.3f} | tvd_from_dev={best['tvd_from_dev']:.3f} | val_total={best['val_total']}")

    return best


def build_random_outer_fold_from_candidate(candidate, fold_index, labels, common_valid_centers):
    outer_name = candidate["fold_name"]
    test_rect = tuple(candidate["rect"])
    test_buffer_rect = tuple(candidate["buffer_rect"])

    test_mask = rect_to_mask(test_rect, rows, cols) & common_valid_centers
    test_buffer_mask = rect_to_mask(test_buffer_rect, rows, cols) & common_valid_centers & (~test_mask)
    dev_mask = common_valid_centers & (~test_mask) & (~test_buffer_mask)
    test_counts = class_counts_in_mask(test_mask, labels)

    if count_present_classes(test_counts) < len(valid_classes):
        raise RuntimeError(f"{outer_name}: test split is missing classes: {test_counts}")
    if min(test_counts.values()) < MIN_TEST_PER_CLASS:
        raise RuntimeError(f"{outer_name}: test split has too few pixels in at least one class: {test_counts}")

    if MULTIBLOCK_OUTER_VAL:
        final_found = search_representative_multiblock_validation(
            dev_mask=dev_mask,
            labels=labels,
            fold_name=outer_name,
            seed=SPLIT_SEARCH_SEEDS.get(f"{outer_name}_FINAL", GLOBAL_SEED + 500 + fold_index),
        )
        final_val_position = "representative_multiblock"
        final_val_rect = final_found["rects"]
        final_val_scale = None
    else:
        raise RuntimeError("This split-only script is intended for Option B. Keep MULTIBLOCK_OUTER_VAL=True.")

    exclusion_mask = (test_buffer_mask | final_found["val_buffer_mask"]) & common_valid_centers

    # Test must remain untouched by validation and training.
    assert int((test_mask & final_found["val_mask"]).sum()) == 0
    assert int((test_mask & final_found["train_mask"]).sum()) == 0
    assert int((test_buffer_mask & final_found["val_mask"]).sum()) == 0

    assert_split_ok(
        train_mask=final_found["train_mask"],
        val_mask=final_found["val_mask"],
        test_mask=test_mask,
        val_buffer_mask=final_found["val_buffer_mask"],
        test_buffer_mask=test_buffer_mask,
        split_name=outer_name,
    )

    return {
        "name": outer_name,
        "test_position": "random_class_aware",
        "final_val_position": final_val_position,
        "test_rect": test_rect,
        "test_buffer_rect": test_buffer_rect,
        "test_mask": test_mask,
        "test_buffer_mask": test_buffer_mask,
        "test_counts": test_counts,
        "test_scale": None,
        "dev_mask": dev_mask,
        "final_train_mask": final_found["train_mask"],
        "final_val_mask": final_found["val_mask"],
        "final_val_buffer_mask": final_found["val_buffer_mask"],
        "final_val_rect": final_val_rect,
        "final_val_counts": final_found["val_counts"],
        "final_train_counts": final_found["train_counts"],
        "final_val_scale": final_val_scale,
        "final_val_distribution_tvd_from_dev": final_found.get("tvd_from_dev", None),
        "final_val_max_class_frac": final_found.get("max_class_frac", None),
        "final_exclusion_mask": exclusion_mask,
        "inner_splits": [],
    }


# =========================
# BUILD / SAVE / LOAD CACHE
# =========================
def split_config_dict():
    return {
        "GLOBAL_SEED": GLOBAL_SEED,
        "MAX_PATCH_SIZE": MAX_PATCH_SIZE,
        "MAX_R": MAX_R,
        "GAP_PIXELS": GAP_PIXELS,
        "valid_classes": valid_classes.tolist(),
        "MIN_CLASSES_REQUIRED": MIN_CLASSES_REQUIRED,
        "MIN_SPLIT_TRAIN_PIXELS_FRAC": MIN_SPLIT_TRAIN_PIXELS_FRAC,
        "MIN_VAL_PER_CLASS": MIN_VAL_PER_CLASS,
        "MIN_TRAIN_PER_CLASS": MIN_TRAIN_PER_CLASS,
        "STRICT_CLASS_COVERAGE": STRICT_CLASS_COVERAGE,
        "N_RANDOM_OUTER_FOLDS": N_RANDOM_OUTER_FOLDS,
        "MIN_TEST_PER_CLASS": MIN_TEST_PER_CLASS,
        "N_RANDOM_CANDIDATES": N_RANDOM_CANDIDATES,
        "CANDIDATE_KEEP_TOP": CANDIDATE_KEEP_TOP,
        "COMBINATION_TRIALS": COMBINATION_TRIALS,
        "TEST_AREA_FRAC_RANGE": TEST_AREA_FRAC_RANGE,
        "ASPECT_RATIO_RANGE": ASPECT_RATIO_RANGE,
        "TARGET_RARE_CLASS": TARGET_RARE_CLASS,
        "TARGETED_RARE_CLASS_FRACTION": TARGETED_RARE_CLASS_FRACTION,
        "MAX_TEST_RECT_OVERLAP_FRAC": MAX_TEST_RECT_OVERLAP_FRAC,
        "MULTIBLOCK_OUTER_VAL": MULTIBLOCK_OUTER_VAL,
        "OUTER_VAL_N_BLOCKS": OUTER_VAL_N_BLOCKS,
        "OUTER_VAL_TOTAL_AREA_FRAC": OUTER_VAL_TOTAL_AREA_FRAC,
        "OUTER_VAL_BLOCK_AREA_FRAC": OUTER_VAL_BLOCK_AREA_FRAC,
        "OUTER_VAL_BLOCK_H_FRAC": OUTER_VAL_BLOCK_H_FRAC,
        "OUTER_VAL_RANDOM_CENTERS": OUTER_VAL_RANDOM_CENTERS,
        "OUTER_VAL_GRID_N": OUTER_VAL_GRID_N,
        "OUTER_VAL_TOP_CANDIDATES": OUTER_VAL_TOP_CANDIDATES,
        "OUTER_VAL_COMBO_TRIALS": OUTER_VAL_COMBO_TRIALS,
        "OUTER_VAL_MAX_SINGLE_CLASS_FRAC": OUTER_VAL_MAX_SINGLE_CLASS_FRAC,
        "OUTER_VAL_MAX_TVD_FROM_DEV": OUTER_VAL_MAX_TVD_FROM_DEV,
        "OUTER_VAL_MIN_TOTAL_PIXELS": OUTER_VAL_MIN_TOTAL_PIXELS,
        "OUTER_VAL_MIN_PER_CLASS": OUTER_VAL_MIN_PER_CLASS,
        "OUTER_TRAIN_MIN_PER_CLASS": OUTER_TRAIN_MIN_PER_CLASS,
    }


def build_outer_folds(labels):
    stage("4/7 BUILD RANDOM CLASS-AWARE 4-OUTER SPATIAL SPLIT")

    common_valid_centers = build_common_valid_centers(
        rows=rows,
        cols=cols,
        label_mask=label_mask,
        max_patch_size=MAX_PATCH_SIZE,
        gap_pixels=GAP_PIXELS,
    )

    class_integrals = {
        int(cls): make_integral(common_valid_centers & (labels == int(cls)))
        for cls in valid_classes
    }

    candidate_rects = generate_candidate_test_rectangles(
        labels=labels,
        common_valid_centers=common_valid_centers,
        class_integrals=class_integrals,
    )

    selected_random_tests = select_best_random_outer_set(candidate_rects)

    outer_folds = []
    for idx, candidate in enumerate(selected_random_tests, 1):
        fold = build_random_outer_fold_from_candidate(candidate, idx, labels, common_valid_centers)
        outer_folds.append(fold)
        print(f"BUILT {fold['name']} | test_counts={fold['test_counts']}")

    union_outer_test_mask = np.zeros((rows, cols), dtype=bool)
    union_outer_test_buffer_mask = np.zeros((rows, cols), dtype=bool)
    for fold in outer_folds:
        union_outer_test_mask |= fold["test_mask"]
        union_outer_test_buffer_mask |= fold["test_buffer_mask"]

    union_outer_test_and_buffer_mask = (union_outer_test_mask | union_outer_test_buffer_mask) & common_valid_centers
    global_hpo_safe_mask = common_valid_centers & (~union_outer_test_and_buffer_mask)

    return common_valid_centers, outer_folds, global_hpo_safe_mask



def mask_area_percentage(mask, denominator):
    """Return mask area as percentage of denominator pixels."""
    denominator = max(1, int(denominator))
    return 100.0 * float(np.asarray(mask, dtype=bool).sum()) / float(denominator)


def fold_area_percentage_summary(fold, common_valid_centers=None):
    """
    Area percentages for one outer fold.

    pct_of_full_image uses rows * cols as denominator.
    pct_of_valid_centers uses the common valid labeled center pixels as denominator.
    The values correspond to the actual colored masks, not the raw rectangle extent.
    """
    full_den = int(rows * cols)
    valid_den = int(np.asarray(common_valid_centers, dtype=bool).sum()) if common_valid_centers is not None else full_den

    masks = {
        "train": fold["final_train_mask"],
        "validation": fold["final_val_mask"],
        "test": fold["test_mask"],
        "test_buffer": fold["test_buffer_mask"],
        "validation_buffer": fold["final_val_buffer_mask"],
    }
    masks["all_buffers"] = masks["test_buffer"] | masks["validation_buffer"]
    masks["train_validation_test"] = masks["train"] | masks["validation"] | masks["test"]
    masks["colored_total"] = masks["train"] | masks["validation"] | masks["test"] | masks["all_buffers"]

    out = {}
    for name, mask in masks.items():
        n = int(np.asarray(mask, dtype=bool).sum())
        out[name] = {
            "pixels": n,
            "pct_of_full_image": mask_area_percentage(mask, full_den),
            "pct_of_valid_centers": mask_area_percentage(mask, valid_den),
        }
    return out


def fold_area_percentage_title(fold, common_valid_centers=None):
    """Compact figure title suffix with the main three split percentages."""
    s = fold_area_percentage_summary(fold, common_valid_centers)
    return (
        f"Train {s['train']['pct_of_full_image']:.2f}% | "
        f"Val {s['validation']['pct_of_full_image']:.2f}% | "
        f"Test {s['test']['pct_of_full_image']:.2f}% of full image"
    )

def save_split_cache(common_valid_centers, outer_folds, global_hpo_safe_mask, masks_path, meta_path):
    arrays = {
        "common_valid_centers": common_valid_centers.astype(bool),
        "global_hpo_safe_mask": global_hpo_safe_mask.astype(bool),
    }

    folds_meta = []
    for i, fold in enumerate(outer_folds, 1):
        prefix = f"fold_{i}"

        arrays[f"{prefix}_test_mask"] = fold["test_mask"].astype(bool)
        arrays[f"{prefix}_test_buffer_mask"] = fold["test_buffer_mask"].astype(bool)
        arrays[f"{prefix}_dev_mask"] = fold["dev_mask"].astype(bool)
        arrays[f"{prefix}_final_train_mask"] = fold["final_train_mask"].astype(bool)
        arrays[f"{prefix}_final_val_mask"] = fold["final_val_mask"].astype(bool)
        arrays[f"{prefix}_final_val_buffer_mask"] = fold["final_val_buffer_mask"].astype(bool)
        arrays[f"{prefix}_final_exclusion_mask"] = fold["final_exclusion_mask"].astype(bool)

        folds_meta.append({
            "index": i,
            "name": fold["name"],
            "test_position": fold.get("test_position", None),
            "final_val_position": fold.get("final_val_position", None),
            "test_rect": fold.get("test_rect", None),
            "test_buffer_rect": fold.get("test_buffer_rect", None),
            "final_val_rect": fold.get("final_val_rect", None),
            "test_counts": fold.get("test_counts", None),
            "final_val_counts": fold.get("final_val_counts", None),
            "final_train_counts": fold.get("final_train_counts", None),
            "final_val_distribution_tvd_from_dev": fold.get("final_val_distribution_tvd_from_dev", None),
            "final_val_max_class_frac": fold.get("final_val_max_class_frac", None),
            "area_percentages": fold_area_percentage_summary(fold, common_valid_centers),
        })

    metadata = {
        "split_cache_name": SPLIT_CACHE_NAME,
        "created_unix": time.time(),
        "rows": int(rows),
        "cols": int(cols),
        "stack_path": stack_path,
        "ref_path": ref_path,
        "valid_classes": valid_classes.tolist(),
        "class_names": CLASS_NAMES,
        "config": split_config_dict(),
        "folds": folds_meta,
    }

    np.savez_compressed(masks_path, **arrays)
    save_json(metadata, meta_path)

    print(f"✅ Saved masks: {masks_path}")
    print(f"✅ Saved metadata: {meta_path}")


def load_split_cache(labels, masks_path, meta_path):
    if not os.path.exists(masks_path):
        raise FileNotFoundError(f"Missing cached split masks: {masks_path}")
    if not os.path.exists(meta_path):
        raise FileNotFoundError(f"Missing cached split metadata: {meta_path}")

    with open(meta_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    if int(metadata["rows"]) != int(rows) or int(metadata["cols"]) != int(cols):
        raise RuntimeError(
            f"Cached split shape mismatch. Cache={metadata['rows']}x{metadata['cols']}, current={rows}x{cols}"
        )

    data = np.load(masks_path, allow_pickle=False)
    common_valid_centers = data["common_valid_centers"].astype(bool)
    global_hpo_safe_mask = data["global_hpo_safe_mask"].astype(bool) if "global_hpo_safe_mask" in data else None

    outer_folds = []
    for fold_meta in metadata["folds"]:
        i = int(fold_meta["index"])
        prefix = f"fold_{i}"

        test_mask = data[f"{prefix}_test_mask"].astype(bool)
        test_buffer_mask = data[f"{prefix}_test_buffer_mask"].astype(bool)
        dev_mask = data[f"{prefix}_dev_mask"].astype(bool)
        final_train_mask = data[f"{prefix}_final_train_mask"].astype(bool)
        final_val_mask = data[f"{prefix}_final_val_mask"].astype(bool)
        final_val_buffer_mask = data[f"{prefix}_final_val_buffer_mask"].astype(bool)
        final_exclusion_mask = data[f"{prefix}_final_exclusion_mask"].astype(bool) if f"{prefix}_final_exclusion_mask" in data else (test_buffer_mask | final_val_buffer_mask)

        test_counts = class_counts_in_mask(test_mask, labels)
        val_counts = class_counts_in_mask(final_val_mask, labels)
        train_counts = class_counts_in_mask(final_train_mask, labels)

        for split_name, counts in [("train", train_counts), ("validation", val_counts), ("test", test_counts)]:
            if count_present_classes(counts) < len(valid_classes):
                raise RuntimeError(f"Cached {fold_meta['name']} {split_name} is missing classes: {counts}")

        assert_split_ok(
            train_mask=final_train_mask,
            val_mask=final_val_mask,
            test_mask=test_mask,
            val_buffer_mask=final_val_buffer_mask,
            test_buffer_mask=test_buffer_mask,
            split_name=fold_meta["name"],
        )

        outer_folds.append({
            "name": fold_meta["name"],
            "test_position": fold_meta.get("test_position", "cached"),
            "final_val_position": fold_meta.get("final_val_position", "cached"),
            "test_rect": fold_meta.get("test_rect", None),
            "test_buffer_rect": fold_meta.get("test_buffer_rect", None),
            "final_val_rect": fold_meta.get("final_val_rect", None),
            "test_mask": test_mask,
            "test_buffer_mask": test_buffer_mask,
            "dev_mask": dev_mask,
            "final_train_mask": final_train_mask,
            "final_val_mask": final_val_mask,
            "final_val_buffer_mask": final_val_buffer_mask,
            "final_exclusion_mask": final_exclusion_mask,
            "test_counts": test_counts,
            "final_val_counts": val_counts,
            "final_train_counts": train_counts,
            "final_val_distribution_tvd_from_dev": fold_meta.get("final_val_distribution_tvd_from_dev", None),
            "final_val_max_class_frac": fold_meta.get("final_val_max_class_frac", None),
        })

    return common_valid_centers, outer_folds, global_hpo_safe_mask, metadata


# =========================
# AUDITS + VISUALIZATION
# =========================
def counts_record_from_mask(fold_name, split_name, mask, labels, common_valid_centers=None):
    counts = class_counts_in_mask(mask, labels)
    mask_pixels = int(np.asarray(mask, dtype=bool).sum())
    full_den = int(rows * cols)
    valid_den = int(np.asarray(common_valid_centers, dtype=bool).sum()) if common_valid_centers is not None else full_den
    record = {
        "fold": str(fold_name),
        "split": str(split_name),
        "mask_pixels": mask_pixels,
        "total_class_pixels": int(sum(counts.values())),
        "pct_of_full_image": float(100.0 * mask_pixels / max(1, full_den)),
        "pct_of_valid_centers": float(100.0 * mask_pixels / max(1, valid_den)),
        "present_classes": int(count_present_classes(counts)),
        "min_class_pixels": int(min(counts.values())) if len(counts) else 0,
    }
    for cls in valid_classes:
        record[f"class_{int(cls)}_{CLASS_NAMES[int(cls)]}"] = int(counts.get(int(cls), 0))
    return record


def export_outer_split_class_counts(outer_folds, labels, paths, common_valid_centers=None):
    rows_out = []
    for fold in outer_folds:
        rows_out.append(counts_record_from_mask(fold["name"], "train", fold["final_train_mask"], labels, common_valid_centers))
        rows_out.append(counts_record_from_mask(fold["name"], "validation", fold["final_val_mask"], labels, common_valid_centers))
        rows_out.append(counts_record_from_mask(fold["name"], "test", fold["test_mask"], labels, common_valid_centers))
        rows_out.append(counts_record_from_mask(fold["name"], "test_buffer", fold["test_buffer_mask"], labels, common_valid_centers))
        rows_out.append(counts_record_from_mask(fold["name"], "validation_buffer", fold["final_val_buffer_mask"], labels, common_valid_centers))

        all_buffers = fold["test_buffer_mask"] | fold["final_val_buffer_mask"]
        rows_out.append(counts_record_from_mask(fold["name"], "all_buffers", all_buffers, labels, common_valid_centers))

        colored_total = fold["final_train_mask"] | fold["final_val_mask"] | fold["test_mask"] | all_buffers
        rows_out.append(counts_record_from_mask(fold["name"], "colored_total", colored_total, labels, common_valid_centers))

    df = pd.DataFrame(rows_out)

    for path in paths:
        df.to_csv(path, index=False)
        print(f"✅ Split class-count + area-percent audit saved: {path}")

    area_summary = df[df["split"].isin(["train", "validation", "test", "all_buffers", "colored_total"])][
        ["fold", "split", "mask_pixels", "pct_of_full_image", "pct_of_valid_centers"]
    ].copy()
    area_summary["pct_of_full_image"] = area_summary["pct_of_full_image"].round(4)
    area_summary["pct_of_valid_centers"] = area_summary["pct_of_valid_centers"].round(4)

    area_paths = []
    for path in paths:
        root, ext = os.path.splitext(path)
        area_path = f"{root}_area_percentages.csv"
        area_summary.to_csv(area_path, index=False)
        area_paths.append(area_path)
        print(f"✅ Split area percentages saved: {area_path}")

    print("\n[OUTER SPLIT AREA PERCENTAGES]")
    print(area_summary.to_string(index=False))

    critical = df[df["split"].isin(["train", "validation", "test"])]
    bad = critical[critical["present_classes"] < len(valid_classes)]
    if len(bad) > 0:
        raise RuntimeError(
            "Reviewer-safe split failed: at least one train/validation/test split is missing a class.\n"
            + bad.to_string(index=False)
        )

    return df


LAYOUT_CMAP = ListedColormap(["white", "royalblue", "limegreen", "crimson", "gold"])
LEGEND_ITEMS = [
    Patch(facecolor="royalblue", edgecolor="black", label="Training area"),
    Patch(facecolor="limegreen", edgecolor="black", label="Validation area"),
    Patch(facecolor="crimson", edgecolor="black", label="Test area"),
    Patch(facecolor="gold", edgecolor="black", label="Buffer / dead zone"),
]


def save_outer_layout_figure(outer_folds, fig_path, common_valid_centers=None):
    fig, axes = plt.subplots(2, 3, figsize=(22, 14))
    axes = np.asarray(axes).ravel()

    for i, fold in enumerate(outer_folds):
        vis = np.zeros((rows, cols), dtype=np.uint8)
        vis[fold["final_train_mask"]] = 1
        vis[fold["final_val_mask"]] = 2
        vis[fold["test_mask"]] = 3
        vis[fold["test_buffer_mask"] | fold["final_val_buffer_mask"]] = 4

        ax = axes[i]
        ax.imshow(vis, cmap=LAYOUT_CMAP, vmin=0, vmax=4)
        ax.axis("off")
        ax.set_title(f"{fold['name']}\n{fold_area_percentage_title(fold, common_valid_centers)}", fontsize=10)

    for j in range(len(outer_folds), len(axes)):
        axes[j].axis("off")

    fig.legend(handles=LEGEND_ITEMS, loc="lower center", ncol=4, frameon=True)
    plt.tight_layout(rect=(0, 0.06, 1, 0.96))
    plt.savefig(fig_path, dpi=300)
    plt.close(fig)
    print(f"✅ Layout figure saved: {fig_path}")


def save_individual_fold_figures(outer_folds, fig_dir, common_valid_centers=None):
    for fold in outer_folds:
        vis = np.zeros((rows, cols), dtype=np.uint8)
        vis[fold["final_train_mask"]] = 1
        vis[fold["final_val_mask"]] = 2
        vis[fold["test_mask"]] = 3
        vis[fold["test_buffer_mask"] | fold["final_val_buffer_mask"]] = 4

        fig, ax = plt.subplots(figsize=(10, 10))
        ax.imshow(vis, cmap=LAYOUT_CMAP, vmin=0, vmax=4)
        ax.axis("off")
        ax.set_title(f"{fold['name']}\n{fold_area_percentage_title(fold, common_valid_centers)}", fontsize=10)
        fig.legend(handles=LEGEND_ITEMS, loc="lower center", ncol=4, frameon=True)
        plt.tight_layout(rect=(0, 0.08, 1, 0.96))

        path = os.path.join(fig_dir, f"{fold['name']}_layout.png")
        plt.savefig(path, dpi=300)
        plt.close(fig)
        print(f"✅ Individual fold figure saved: {path}")


def export_mask_geotiffs(outer_folds, out_dir):
    """
    Optional: exports one categorical GeoTIFF per fold:
    0 background, 1 train, 2 validation, 3 test, 4 buffer.
    """
    if not EXPORT_MASK_GEOTIFFS:
        return

    profile = stack_profile.copy()
    profile.update({
        "count": 1,
        "dtype": "uint8",
        "nodata": 0,
        "compress": "lzw",
    })

    tif_dir = os.path.join(out_dir, "mask_geotiffs")
    os.makedirs(tif_dir, exist_ok=True)

    for fold in outer_folds:
        vis = np.zeros((rows, cols), dtype=np.uint8)
        vis[fold["final_train_mask"]] = 1
        vis[fold["final_val_mask"]] = 2
        vis[fold["test_mask"]] = 3
        vis[fold["test_buffer_mask"] | fold["final_val_buffer_mask"]] = 4

        path = os.path.join(tif_dir, f"{fold['name']}_split_layout.tif")
        with rasterio.open(path, "w", **profile) as dst:
            dst.write(vis, 1)
        print(f"✅ Mask GeoTIFF saved: {path}")


def compute_and_export_normalization_stats(outer_folds):
    """
    Optional: mirrors the old full pipeline's per-fold normalization JSONs.
    This reads the full stack, so it is OFF by default in this split-only script.
    """
    if not EXPORT_NORMALIZATION_STATS:
        return

    stage("OPTIONAL: EXPORT NORMALIZATION STATS")
    with rasterio.open(stack_path) as src:
        stack = src.read().astype(np.float32)
    stack = np.nan_to_num(stack, nan=0.0, posinf=0.0, neginf=0.0)
    stack = np.moveaxis(stack, 0, -1).astype(np.float32)

    def compute_band_stats(stack_arr, train_mask, max_pixels=100000, seed=42):
        coords = np.argwhere(train_mask)
        if len(coords) == 0:
            raise ValueError("Cannot compute band stats: empty train mask.")
        rng = np.random.default_rng(int(seed))
        idx = rng.choice(len(coords), size=min(int(max_pixels), len(coords)), replace=False)
        pixels = stack_arr[coords[idx][:, 0], coords[idx][:, 1], :]
        pixels = np.nan_to_num(pixels, nan=0.0, posinf=0.0, neginf=0.0)
        mean = np.nan_to_num(pixels.mean(axis=0), nan=0.0, posinf=0.0, neginf=0.0)
        std = np.nan_to_num(pixels.std(axis=0), nan=1.0, posinf=1.0, neginf=1.0)
        std[std == 0] = 1.0
        return mean.astype(np.float32), std.astype(np.float32)

    for fold_idx, fold in enumerate(outer_folds, 1):
        band_mean, band_std = compute_band_stats(stack, fold["final_train_mask"], seed=GLOBAL_SEED + fold_idx)
        payload = {
            "fold": fold["name"],
            "band_names": band_names,
            "band_mean": band_mean.tolist(),
            "band_std": band_std.tolist(),
        }
        path = os.path.join(out_dir, f"normalization_stats_{fold['name']}.json")
        save_json(payload, path)
        print(f"✅ Normalization stats saved: {path}")





## 4. Experiment configuration and TensorFlow setup


In [ ]:
# ============================================================
# MODEL / PIPELINE CONFIG
# ============================================================
# Split cache behavior.
# OVERWRITE_EXISTING_SPLIT is already defined above. Keep False for reproducibility.
BUILD_OR_LOAD_SPLIT = True

# HPO split cache. These are spatial validation blocks used ONLY for parameter selection.
HPO_SPLIT_CACHE_NAME = f"{SPLIT_CACHE_NAME}_global_hpo4_v1"
HPO_SPLIT_MASKS_PATH = os.path.join(split_cache_dir, f"{HPO_SPLIT_CACHE_NAME}_masks.npz")
HPO_SPLIT_META_PATH  = os.path.join(split_cache_dir, f"{HPO_SPLIT_CACHE_NAME}_metadata.json")
REBUILD_HPO_SPLITS = False
BUILD_HPO_SPLITS_ONLY_WHEN_NEEDED = True

# Main model outputs.
out_dir = "/content/kerkini_optionB_all_models_hpo_final_v1"
fig_dir = os.path.join(out_dir, "figures")
model_dir = os.path.join(out_dir, "models")
report_dir = os.path.join(out_dir, "reports")
hpo_dir = os.path.join(out_dir, "hpo")
for _d in [out_dir, fig_dir, model_dir, report_dir, hpo_dir]:
    os.makedirs(_d, exist_ok=True)

# Architecture switches. Turn any of these False to disable that architecture completely.
ARCHITECTURE_ENABLED = {
    "CNN": False,       # already completed; keep disabled for the continuation run
    "ViT": False,
    "UNetPP": False,
    "ConvNeXt": True,
}

# HPO and final-run switches. Useful when you already ran some parts.
RUN_HPO_BY_ARCH = {
    "CNN": False,       # do not rerun CNN HPO
    "ViT": False,
    "UNetPP": False,
    "ConvNeXt": True,
}
RUN_FINAL_BY_ARCH = {
    "CNN": False,       # do not rerun CNN final evaluation
    "ViT": False,
    "UNetPP": False,
    "ConvNeXt": True,
}

# If True and best_global_config_<ARCH>.json exists, HPO is skipped and that config is used.
SKIP_HPO_IF_BEST_CONFIG_EXISTS = True

# If True and final summary exists, final training/evaluation for that architecture is skipped.
SKIP_FINAL_IF_SUMMARY_EXISTS = True

# If summary does not exist but per-fold predictions exist, reuse them instead of retraining that fold.
SKIP_FOLD_IF_PREDICTIONS_EXIST = True

# Logging/checkpoint policy. HPO checkpoints can use disk, but they make the run fully auditable.
SAVE_HPO_CHECKPOINTS = True
SAVE_HPO_TRAINING_CURVES = True
SAVE_MODEL_SUMMARIES = True
SAVE_TIMING_LOGS = True

STOP_ON_MODEL_ERROR = True

# Continuation mode: use the existing split/HPO split caches and append remaining architectures.
# Keep REBUILD_HPO_SPLITS=False and OVERWRITE_EXISTING_SPLIT=False for exact comparability.

# Compute / training settings.
USE_MIXED_PRECISION = True
USE_XLA = True
A100_FAST_MODE = True
USE_PREEXTRACTED_PATCH_ARRAYS = True
PATCH_ARRAY_DTYPE = np.float16

N_PARAM_TRIALS = 8
HPO_EPOCH_CAP = 50
# If False, every HPO trial may train up to HPO_EPOCH_CAP.
# If True, each trial is capped by its own config["epochs"] too.
HPO_USE_TRIAL_EPOCHS = False
FINAL_EPOCHS = 50

SEARCH_TRAIN_PER_CLASS = 3000
SEARCH_VAL_PER_CLASS = 1000
FINAL_TRAIN_PER_CLASS = 15000
FINAL_VAL_PER_CLASS = 4000

HPO_BATCH_SIZE_OVERRIDE = 512
FINAL_BATCH_SIZE_OVERRIDE = 512
VAL_INFER_BATCH_SIZE = 16384
INFER_BATCH_SIZE = 8192
DO_FULL_TEST_INFERENCE = True
TEST_SAMPLE_PER_CLASS_IF_NOT_FULL = 10000

EARLY_STOPPING_PATIENCE_HPO = 4
REDUCE_LR_PATIENCE_HPO = 2
EARLY_STOPPING_PATIENCE_FINAL = 4
EARLY_STOPPING_MIN_DELTA = 1e-4
REDUCE_LR_PATIENCE_FINAL = 3

ENABLE_MEMORY_GUARD = True
RAM_HARD_LIMIT_FRAC = 0.98

# Inner/global spatial HPO geometry. Kept from your previous pipeline.
INNER_VAL_AREA_FRAC = 0.10
INNER_VAL_H_FRAC = 0.32
MIN_HPO_VAL_PER_CLASS = 100
MIN_HPO_TRAIN_PER_CLASS = 300

# Your fixed HPO trial list. This is the search space actually evaluated, not a full Cartesian grid.
# Trial 3 matches the U-Net++ fixed configuration you used before.
param_configs = [
    {"trial": 1, "patch_size": 19, "filters1": 32, "filters2": 64,  "dense_units": 256, "dropout": 0.30, "lr": 3e-4, "batch_size": 256, "epochs": 20},
    {"trial": 2, "patch_size": 23, "filters1": 48, "filters2": 96,  "dense_units": 384, "dropout": 0.30, "lr": 2e-4, "batch_size": 256, "epochs": 25},
    {"trial": 3, "patch_size": 27, "filters1": 64, "filters2": 128, "dense_units": 512, "dropout": 0.40, "lr": 1e-4, "batch_size": 256, "epochs": 25},
    {"trial": 4, "patch_size": 23, "filters1": 64, "filters2": 96,  "dense_units": 256, "dropout": 0.20, "lr": 3e-4, "batch_size": 256, "epochs": 20},
    {"trial": 5, "patch_size": 19, "filters1": 48, "filters2": 128, "dense_units": 384, "dropout": 0.30, "lr": 2e-4, "batch_size": 256, "epochs": 20},
    {"trial": 6, "patch_size": 27, "filters1": 48, "filters2": 96,  "dense_units": 512, "dropout": 0.20, "lr": 2e-4, "batch_size": 256, "epochs": 25},
    {"trial": 7, "patch_size": 31, "filters1": 32, "filters2": 96,  "dense_units": 256, "dropout": 0.40, "lr": 1e-4, "batch_size": 256, "epochs": 20},
    {"trial": 8, "patch_size": 23, "filters1": 32, "filters2": 128, "dense_units": 512, "dropout": 0.30, "lr": 3e-4, "batch_size": 256, "epochs": 20},
]

# Used only if HPO is disabled and no best config file exists.
MANUAL_FALLBACK_CONFIG_BY_ARCH = {
    "CNN": param_configs[2],
    "ViT": param_configs[2],
    "UNetPP": param_configs[2],
    "ConvNeXt": param_configs[2],
}

NUM_CLASSES = len(valid_classes)
label_to_index = {int(cls): i for i, cls in enumerate(valid_classes)}
index_to_label = {i: int(cls) for cls, i in label_to_index.items()}

os.environ["PYTHONHASHSEED"] = str(GLOBAL_SEED)
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
tf.random.set_seed(GLOBAL_SEED)

# ============================================================
# GPU / TF SETUP
# ============================================================
stage("0/12 GPU / TENSORFLOW SETUP")
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled.")
    except Exception as e:
        print("Could not enable GPU memory growth:", e)

    if USE_MIXED_PRECISION:
        try:
            from tensorflow.keras import mixed_precision
            mixed_precision.set_global_policy("mixed_float16")
            print("Mixed precision enabled: mixed_float16")
        except Exception as e:
            print("Could not enable mixed precision:", e)
    if USE_XLA:
        try:
            tf.config.optimizer.set_jit(True)
            print("TensorFlow XLA JIT enabled.")
        except Exception as e:
            print("Could not enable XLA:", e)
else:
    print("No GPU detected. This will be slow.")

# ============================================================


## 5. Patch extraction, data generators and evaluation helpers


In [ ]:
# GENERAL HELPERS FOR MODELING
# ============================================================
def write_text(path, text):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def append_text(path, text):
    with open(path, "a", encoding="utf-8") as f:
        f.write(text)


def clear_mem(sleep_sec=0.0):
    plt.close("all")
    try:
        tf.keras.backend.clear_session()
    except Exception:
        pass
    gc.collect()
    gc.collect()
    if sleep_sec > 0:
        time.sleep(sleep_sec)


def get_ram_fraction():
    if psutil is None:
        return 0.0
    try:
        return float(psutil.virtual_memory().percent) / 100.0
    except Exception:
        return 0.0


class MemoryGuardCallback(callbacks.Callback):
    def __init__(self, phase_prefix=""):
        super().__init__()
        self.phase_prefix = phase_prefix
    def on_epoch_end(self, epoch, logs=None):
        if not ENABLE_MEMORY_GUARD:
            return
        if get_ram_fraction() >= RAM_HARD_LIMIT_FRAC:
            print(f"[RAM GUARD] Hard RAM limit reached during {self.phase_prefix}. Stopping model training.")
            self.model.stop_training = True


class ProgressEpochPrinter(callbacks.Callback):
    def __init__(self, prefix=""):
        super().__init__()
        self.prefix = prefix
    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_t0 = time.time()
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        elapsed = time.time() - getattr(self, "epoch_t0", time.time())
        msg = f"[{self.prefix}] Epoch {epoch + 1} | {elapsed:.1f}s"
        for key in ["loss", "accuracy", "val_loss", "val_accuracy"]:
            if key in logs:
                msg += f" | {key}={logs[key]:.4f}"
        print(msg)


def load_full_stack_array():
    stage("LOAD FULL STACK ARRAY FOR MODELING")
    with rasterio.open(stack_path) as src:
        arr = src.read().astype(np.float32)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = np.moveaxis(arr, 0, -1).astype(np.float32)
    print("Stack shape (rows, cols, bands):", arr.shape)
    return arr


def sample_balanced_coords(mask, labels, per_class, seed=42):
    rng = np.random.default_rng(int(seed))
    coords = get_coords(mask)
    selected = []
    for cls in valid_classes:
        cls_coords = coords[labels[coords[:, 0], coords[:, 1]] == int(cls)]
        if len(cls_coords) == 0:
            raise RuntimeError(f"No pixels found for class {int(cls)} in sampling mask.")
        n = min(int(per_class), len(cls_coords))
        idx = rng.choice(len(cls_coords), size=n, replace=False)
        selected.append(cls_coords[idx])
    selected = np.vstack(selected).astype(np.int32)
    rng.shuffle(selected)
    return selected


def class_counts_from_coords(coords, labels):
    out = {int(c): 0 for c in valid_classes}
    if len(coords) == 0:
        return out
    vals = labels[coords[:, 0], coords[:, 1]]
    for c in valid_classes:
        out[int(c)] = int((vals == int(c)).sum())
    return out


def compute_band_stats(stack_arr, train_mask, max_pixels=100000, seed=42):
    coords = np.argwhere(train_mask)
    if len(coords) == 0:
        raise ValueError("Cannot compute band stats: empty train mask.")
    rng = np.random.default_rng(int(seed))
    idx = rng.choice(len(coords), size=min(int(max_pixels), len(coords)), replace=False)
    pixels = stack_arr[coords[idx][:, 0], coords[idx][:, 1], :]
    pixels = np.nan_to_num(pixels, nan=0.0, posinf=0.0, neginf=0.0)
    mean = np.nan_to_num(pixels.mean(axis=0), nan=0.0, posinf=0.0, neginf=0.0)
    std = np.nan_to_num(pixels.std(axis=0), nan=1.0, posinf=1.0, neginf=1.0)
    std[std == 0] = 1.0
    return mean.astype(np.float32), std.astype(np.float32)


def extract_patch_array(image, labels, coords, patch_size, band_mean, band_std, dtype=np.float16, progress_name=""):
    coords = np.asarray(coords, dtype=np.int32)
    n = len(coords)
    r = int(patch_size) // 2
    channels = int(image.shape[-1])
    X = np.empty((n, int(patch_size), int(patch_size), channels), dtype=dtype)
    y = np.empty((n,), dtype=np.int32)
    band_mean = np.asarray(band_mean, dtype=np.float32).reshape((1, 1, -1))
    band_std = np.asarray(band_std, dtype=np.float32).reshape((1, 1, -1))

    t0 = time.time()
    for i, (rr, cc) in enumerate(coords):
        patch = np.asarray(image[rr-r:rr+r+1, cc-r:cc+r+1, :], dtype=np.float32)
        patch = np.nan_to_num(patch, nan=0.0, posinf=0.0, neginf=0.0)
        X[i] = np.nan_to_num((patch - band_mean) / band_std, nan=0.0, posinf=0.0, neginf=0.0).astype(dtype, copy=False)
        y[i] = label_to_index[int(labels[rr, cc])]
        if progress_name and (i + 1) % 25000 == 0:
            print(f"[{progress_name}] extracted {i+1}/{n} patches ({time.time()-t0:.1f}s)")
    return X, y


class PatchSequence(Sequence):
    def __init__(self, image, labels, coords, patch_size, band_mean, band_std, batch_size=32, shuffle=True):
        self.image = image
        self.labels = labels
        self.coords = np.asarray(coords, dtype=np.int32)
        self.patch_size = int(patch_size)
        self.r = self.patch_size // 2
        self.band_mean = np.asarray(band_mean, dtype=np.float32).reshape((1, 1, -1))
        self.band_std = np.asarray(band_std, dtype=np.float32).reshape((1, 1, -1))
        self.batch_size = int(batch_size)
        self.shuffle = shuffle
        self.indices = np.arange(len(self.coords))
        self.on_epoch_end()
    def __len__(self):
        return int(np.ceil(len(self.coords) / self.batch_size)) if len(self.coords) > 0 else 0
    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_coords = self.coords[batch_idx]
        X = np.empty((len(batch_coords), self.patch_size, self.patch_size, self.image.shape[-1]), dtype=np.float32)
        y = np.empty((len(batch_coords),), dtype=np.int32)
        for i, (rr, cc) in enumerate(batch_coords):
            patch = np.asarray(self.image[rr-self.r:rr+self.r+1, cc-self.r:cc+self.r+1, :], dtype=np.float32)
            patch = np.nan_to_num(patch, nan=0.0, posinf=0.0, neginf=0.0)
            X[i] = np.nan_to_num((patch - self.band_mean) / self.band_std, nan=0.0, posinf=0.0, neginf=0.0)
            y[i] = label_to_index[int(self.labels[rr, cc])]
        return X, y
    def on_epoch_end(self):
        if self.shuffle and len(self.indices) > 0:
            np.random.shuffle(self.indices)


def batch_patch_predict(model, image, coords, patch_size, band_mean, band_std, batch_size=8192, progress_name=""):
    coords = np.asarray(coords, dtype=np.int32)
    preds = np.zeros((len(coords),), dtype=np.uint8)
    r = int(patch_size) // 2
    n_total = len(coords)
    if n_total == 0:
        return preds

    band_mean = np.asarray(band_mean, dtype=np.float32).reshape((1, 1, -1))
    band_std = np.asarray(band_std, dtype=np.float32).reshape((1, 1, -1))

    current_bs = max(1, int(batch_size))
    start_idx = 0
    t0 = time.time()
    while start_idx < n_total:
        end_idx = min(n_total, start_idx + current_bs)
        batch_coords = coords[start_idx:end_idx]
        try:
            Xb = np.empty((len(batch_coords), patch_size, patch_size, image.shape[-1]), dtype=PATCH_ARRAY_DTYPE)
            for i, (rr, cc) in enumerate(batch_coords):
                patch = np.asarray(image[rr-r:rr+r+1, cc-r:cc+r+1, :], dtype=np.float32)
                patch = np.nan_to_num(patch, nan=0.0, posinf=0.0, neginf=0.0)
                Xb[i] = np.nan_to_num((patch - band_mean) / band_std, nan=0.0, posinf=0.0, neginf=0.0).astype(PATCH_ARRAY_DTYPE, copy=False)
            prob = model.predict(Xb, batch_size=current_bs, verbose=0)
            preds[start_idx:end_idx] = np.array([index_to_label[int(k)] for k in np.argmax(prob, axis=1).astype(np.uint8)], dtype=np.uint8)
            start_idx = end_idx
            if progress_name and (start_idx == n_total or start_idx % max(1, current_bs * 5) == 0):
                print(f"[{progress_name}] predicted {start_idx}/{n_total} pixels ({time.time()-t0:.1f}s)")
        except Exception as e:
            if current_bs <= 1:
                raise e
            current_bs = max(1, current_bs // 2)
            print(f"[{progress_name}] predict batch failed; reducing batch size to {current_bs}")
            clear_mem(0.1)
    return preds


def labels_as_names():
    return [CLASS_NAMES[int(i)] for i in valid_classes]


def confusion_matrix_dataframe(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=list(valid_classes))
    names = labels_as_names()
    return pd.DataFrame(cm, index=[f"true_{n}" for n in names], columns=[f"pred_{n}" for n in names])


def metric_dict(y_true, y_pred, prefix):
    return {
        f"{prefix}_acc": float(accuracy_score(y_true, y_pred)),
        f"{prefix}_balanced_acc": float(balanced_accuracy_score(y_true, y_pred)),
        f"{prefix}_macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        f"{prefix}_weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        f"{prefix}_kappa": float(cohen_kappa_score(y_true, y_pred, labels=list(valid_classes))),
    }


def save_training_curve(history, fig_path, title):
    hist = history.history
    train_loss = hist.get("loss", [])
    val_loss = hist.get("val_loss", [])
    train_acc = hist.get("accuracy", [])
    val_acc = hist.get("val_accuracy", [])

    best_epoch = int(np.argmin(val_loss)) if len(val_loss) else None

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(train_loss, label="train_loss")
    ax.plot(val_loss, label="val_loss")
    if train_acc:
        ax.plot(train_acc, label="train_acc")
    if val_acc:
        ax.plot(val_acc, label="val_acc")
    if best_epoch is not None:
        ax.axvline(best_epoch, linestyle="--", label=f"best_epoch={best_epoch + 1}")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
    plt.tight_layout()
    plt.savefig(fig_path, dpi=300)
    plt.close(fig)


def save_area_curve_panel(fold, history, fig_path, title):
    vis = np.zeros((rows, cols), dtype=np.uint8)
    vis[fold["final_train_mask"]] = 1
    vis[fold["final_val_mask"]] = 2
    vis[fold["test_mask"]] = 3
    vis[fold["test_buffer_mask"] | fold["final_val_buffer_mask"]] = 4

    hist = history.history
    val_loss = hist.get("val_loss", [])
    best_epoch = int(np.argmin(val_loss)) if len(val_loss) else None

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes[0].imshow(vis, cmap=LAYOUT_CMAP, vmin=0, vmax=4, interpolation="nearest")
    axes[0].set_title("Cached spatial areas")
    axes[0].axis("off")

    axes[1].plot(hist.get("loss", []), label="train_loss")
    axes[1].plot(hist.get("val_loss", []), label="val_loss")
    if best_epoch is not None:
        axes[1].axvline(best_epoch, linestyle="--", label=f"best epoch={best_epoch + 1}")
    axes[1].set_title("Training curve")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    fig.suptitle(title)
    fig.legend(handles=LEGEND_ITEMS, loc="lower center", ncol=4, frameon=True)
    plt.tight_layout(rect=(0, 0.08, 1, 0.95))
    plt.savefig(fig_path, dpi=300)
    plt.close(fig)



## 6. Model definitions: CNN, ViT, U-Net++ and ConvNeXt-style


In [ ]:
# ============================================================
# MODEL DEFINITIONS
# ============================================================
def build_standard_cnn(patch_size, input_channels, filters1, filters2, dense_units, dropout, lr):
    inp = layers.Input(shape=(patch_size, patch_size, input_channels), name="input_patch")

    x = layers.Conv2D(filters1, kernel_size=3, padding="same", use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=2)(x)

    x = layers.Conv2D(filters2, kernel_size=3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=2)(x)

    x = layers.Conv2D(filters2 * 2, kernel_size=3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)

    model = models.Model(inp, out, name="Standard_CNN")
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=float(lr), weight_decay=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_vit(patch_size, input_channels, filters1, filters2, dense_units, dropout, lr):
    embed_dim = int(filters2)
    num_heads = 4
    sub_patch_size = 3 if int(patch_size) < 20 else 4

    inp = layers.Input(shape=(patch_size, patch_size, input_channels), name="input_patch")

    x = layers.Conv2D(embed_dim, kernel_size=sub_patch_size, strides=sub_patch_size, padding="valid")(inp)
    num_patches = int(x.shape[1]) * int(x.shape[2])
    x = layers.Reshape((num_patches, embed_dim))(x)

    positions = tf.range(start=0, limit=num_patches, delta=1)
    pos_emb = layers.Embedding(input_dim=num_patches, output_dim=embed_dim)(positions)
    x = x + pos_emb

    for block_idx in range(3):
        x1 = layers.LayerNormalization(epsilon=1e-6, name=f"vit_ln1_{block_idx}")(x)
        attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=max(1, embed_dim // num_heads),
            dropout=float(dropout),
            name=f"vit_mha_{block_idx}",
        )(x1, x1)
        x = layers.Add(name=f"vit_attn_add_{block_idx}")([x, attn])

        x2 = layers.LayerNormalization(epsilon=1e-6, name=f"vit_ln2_{block_idx}")(x)
        mlp = layers.Dense(dense_units, activation=tf.nn.gelu, name=f"vit_mlp1_{block_idx}")(x2)
        mlp = layers.Dropout(dropout, name=f"vit_mlp_dropout1_{block_idx}")(mlp)
        mlp = layers.Dense(embed_dim, name=f"vit_mlp2_{block_idx}")(mlp)
        mlp = layers.Dropout(dropout, name=f"vit_mlp_dropout2_{block_idx}")(mlp)
        x = layers.Add(name=f"vit_mlp_add_{block_idx}")([x, mlp])

    x = layers.LayerNormalization(epsilon=1e-6, name="vit_final_ln")(x)
    x = layers.GlobalAveragePooling1D(name="vit_gap")(x)
    x = layers.Dropout(dropout, name="vit_head_dropout")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="class_output")(x)

    model = models.Model(inp, out, name="ViT_patch_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=float(lr), weight_decay=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def conv_block(x, filters, dropout=0.0, name="conv"):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv1")(x)
    x = layers.BatchNormalization(name=f"{name}_bn1")(x)
    x = layers.Activation("relu", name=f"{name}_relu1")(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv2")(x)
    x = layers.BatchNormalization(name=f"{name}_bn2")(x)
    x = layers.Activation("relu", name=f"{name}_relu2")(x)
    if dropout and dropout > 0:
        x = layers.SpatialDropout2D(min(float(dropout), 0.5), name=f"{name}_spatial_dropout")(x)
    return x


def resize_like(x, ref, name):
    target_h = int(ref.shape[1])
    target_w = int(ref.shape[2])
    return layers.Resizing(target_h, target_w, interpolation="bilinear", name=name)(x)


def build_unetpp_patch_classifier(patch_size, input_channels, filters1, filters2, dense_units, dropout, lr):
    inp = layers.Input(shape=(patch_size, patch_size, input_channels), name="input_patch")

    x00 = conv_block(inp, filters1, dropout=0.0, name="x00")
    p0 = layers.MaxPooling2D(pool_size=2, padding="same", name="pool0")(x00)

    x10 = conv_block(p0, filters2, dropout=0.0, name="x10")
    p1 = layers.MaxPooling2D(pool_size=2, padding="same", name="pool1")(x10)

    x20 = conv_block(p1, filters2 * 2, dropout=dropout * 0.5, name="x20")

    u10 = resize_like(x10, x00, name="up10_to_x00")
    x01 = layers.Concatenate(name="cat_x01")([x00, u10])
    x01 = conv_block(x01, filters1, dropout=dropout * 0.5, name="x01")

    u20 = resize_like(x20, x10, name="up20_to_x10")
    x11 = layers.Concatenate(name="cat_x11")([x10, u20])
    x11 = conv_block(x11, filters2, dropout=dropout * 0.5, name="x11")

    u11 = resize_like(x11, x00, name="up11_to_x00")
    x02 = layers.Concatenate(name="cat_x02")([x00, x01, u11])
    x02 = conv_block(x02, filters1, dropout=dropout, name="x02")

    x = layers.GlobalAveragePooling2D(name="gap_decoder")(x02)
    x = layers.Dense(dense_units, activation="relu", name="dense_classifier")(x)
    x = layers.Dropout(dropout, name="dropout_classifier")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="class_output")(x)

    model = models.Model(inp, out, name="UNetPP_patch_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=float(lr), weight_decay=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def convnext_block(x, dim, dropout=0.0, name="convnext_block"):
    shortcut = x
    x = layers.DepthwiseConv2D(kernel_size=7, padding="same", name=f"{name}_dwconv")(x)
    x = layers.LayerNormalization(epsilon=1e-6, name=f"{name}_ln")(x)
    x = layers.Dense(4 * dim, activation=tf.nn.gelu, name=f"{name}_pw1")(x)
    x = layers.Dropout(dropout, name=f"{name}_drop1")(x)
    x = layers.Dense(dim, name=f"{name}_pw2")(x)
    if dropout and dropout > 0:
        x = layers.Dropout(dropout * 0.5, name=f"{name}_drop2")(x)
    return layers.Add(name=f"{name}_add")([shortcut, x])


def build_convnext_patch_classifier(patch_size, input_channels, filters1, filters2, dense_units, dropout, lr):
    """
    Small ConvNeXt-style patch classifier adapted for 15-31 px patches.
    It is not ImageNet ConvNeXt; it uses ConvNeXt blocks in a small patch-classification setting.
    """
    inp = layers.Input(shape=(patch_size, patch_size, input_channels), name="input_patch")

    x = layers.Conv2D(filters1, kernel_size=3, strides=1, padding="same", name="stem_conv")(inp)
    x = layers.LayerNormalization(epsilon=1e-6, name="stem_ln")(x)
    x = convnext_block(x, filters1, dropout=dropout * 0.5, name="stage1_block1")
    x = convnext_block(x, filters1, dropout=dropout * 0.5, name="stage1_block2")

    x = layers.Conv2D(filters2, kernel_size=2, strides=2, padding="same", name="downsample1_conv")(x)
    x = layers.LayerNormalization(epsilon=1e-6, name="downsample1_ln")(x)
    x = convnext_block(x, filters2, dropout=dropout * 0.5, name="stage2_block1")
    x = convnext_block(x, filters2, dropout=dropout * 0.5, name="stage2_block2")

    x = layers.Conv2D(filters2 * 2, kernel_size=2, strides=2, padding="same", name="downsample2_conv")(x)
    x = layers.LayerNormalization(epsilon=1e-6, name="downsample2_ln")(x)
    x = convnext_block(x, filters2 * 2, dropout=dropout, name="stage3_block1")
    x = convnext_block(x, filters2 * 2, dropout=dropout, name="stage3_block2")

    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.LayerNormalization(epsilon=1e-6, name="head_ln")(x)
    x = layers.Dense(dense_units, activation=tf.nn.gelu, name="dense_classifier")(x)
    x = layers.Dropout(dropout, name="head_dropout")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="class_output")(x)

    model = models.Model(inp, out, name="ConvNeXt_style_patch_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=float(lr), weight_decay=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


ARCHITECTURE_BUILDERS = {
    "CNN": build_standard_cnn,
    "ViT": build_vit,
    "UNetPP": build_unetpp_patch_classifier,
    "ConvNeXt": build_convnext_patch_classifier,
}



## 7. Spatial hyperparameter-search splitting


In [ ]:
# ============================================================
# HPO SPLIT CACHE
# ============================================================
def compute_union_outer_masks(outer_folds, common_valid_centers):
    union_outer_test_mask = np.zeros((rows, cols), dtype=bool)
    union_outer_test_buffer_mask = np.zeros((rows, cols), dtype=bool)
    for fold in outer_folds:
        union_outer_test_mask |= fold["test_mask"]
        union_outer_test_buffer_mask |= fold["test_buffer_mask"]
    union_outer_test_and_buffer_mask = (union_outer_test_mask | union_outer_test_buffer_mask) & common_valid_centers
    global_hpo_safe_mask = common_valid_centers & (~union_outer_test_and_buffer_mask)
    return union_outer_test_mask, union_outer_test_buffer_mask, global_hpo_safe_mask


def build_all_spatial_global_hpo_splits(global_hpo_safe_mask, union_outer_test_mask, union_outer_test_buffer_mask):
    positions = ["top_left", "top_right", "bottom_left", "bottom_right"]
    splits = []
    print("\n[SPATIAL HPO] Building 4 leakage-safe spatial HPO subfolds.")

    for i, pos in enumerate(positions, 1):
        short = f"S{i}_{pos[:2].upper()}"
        split_name = f"GLOBAL_HPO_SPATIAL_{short}"

        expansion_values = [0, GAP_PIXELS * 2, GAP_PIXELS * 4, GAP_PIXELS * 8]
        split_info = None
        last_error = None
        for exp in expansion_values:
            val_search_mask = make_quadrant_region_mask(pos, global_hpo_safe_mask, rows, cols, expansion=exp)
            if int(val_search_mask.sum()) < 2000:
                continue
            try:
                split_info = search_validation_block(
                    candidate_mask=val_search_mask,
                    labels=ref_aligned,
                    position=pos,
                    area_frac=INNER_VAL_AREA_FRAC,
                    h_frac=INNER_VAL_H_FRAC,
                    split_name=split_name,
                    extra_test_mask=union_outer_test_mask,
                    extra_test_buffer_mask=union_outer_test_buffer_mask,
                    base_mask_for_train_frac=val_search_mask,
                    min_val_pixels=1500,
                    min_val_per_class=MIN_HPO_VAL_PER_CLASS,
                    min_train_per_class=MIN_HPO_TRAIN_PER_CLASS,
                    scales=(1.00, 0.90, 0.80, 0.70, 0.60, 0.50),
                    n_grid=7,
                    seed=1100 + i,
                )
                break
            except Exception as e:
                last_error = e

        if split_info is None:
            raise RuntimeError(f"Failed to create Spatial HPO validation block for {pos}: {last_error}")

        rect = split_info["rect"]
        val_mask = rect_to_mask(rect, rows, cols) & global_hpo_safe_mask
        val_buffer_mask = rect_to_mask(dilate_rect(rect, GAP_PIXELS, rows, cols), rows, cols) & global_hpo_safe_mask & (~val_mask)
        train_mask = global_hpo_safe_mask & (~val_mask) & (~val_buffer_mask)

        train_coords = sample_balanced_coords(train_mask, ref_aligned, SEARCH_TRAIN_PER_CLASS, seed=3000 + i)
        val_coords = sample_balanced_coords(val_mask, ref_aligned, SEARCH_VAL_PER_CLASS, seed=4000 + i)

        splits.append({
            "index": i,
            "name": split_name,
            "short": short,
            "position": pos,
            "seed": 4000 + i,
            "rect": rect,
            "train_mask": train_mask,
            "val_mask": val_mask,
            "val_buffer_mask": val_buffer_mask,
            "train_coords": train_coords,
            "val_coords": val_coords,
        })
        print(f"✅ {split_name} | position={pos} | train samples={len(train_coords)} | val samples={len(val_coords)}")
    return splits


def save_hpo_split_cache(global_hpo_splits, masks_path, meta_path):
    arrays = {}
    split_meta = []
    for split in global_hpo_splits:
        i = int(split["index"])
        prefix = f"hpo_{i}"
        arrays[f"{prefix}_train_mask"] = split["train_mask"].astype(bool)
        arrays[f"{prefix}_val_mask"] = split["val_mask"].astype(bool)
        arrays[f"{prefix}_val_buffer_mask"] = split["val_buffer_mask"].astype(bool)
        arrays[f"{prefix}_train_coords"] = np.asarray(split["train_coords"], dtype=np.int32)
        arrays[f"{prefix}_val_coords"] = np.asarray(split["val_coords"], dtype=np.int32)
        split_meta.append({
            "index": i,
            "name": split["name"],
            "short": split["short"],
            "position": split["position"],
            "seed": split["seed"],
            "rect": split["rect"],
        })
    metadata = {
        "hpo_split_cache_name": HPO_SPLIT_CACHE_NAME,
        "split_cache_name": SPLIT_CACHE_NAME,
        "created_unix": time.time(),
        "rows": int(rows),
        "cols": int(cols),
        "splits": split_meta,
        "config": {
            "INNER_VAL_AREA_FRAC": INNER_VAL_AREA_FRAC,
            "INNER_VAL_H_FRAC": INNER_VAL_H_FRAC,
            "MIN_HPO_VAL_PER_CLASS": MIN_HPO_VAL_PER_CLASS,
            "MIN_HPO_TRAIN_PER_CLASS": MIN_HPO_TRAIN_PER_CLASS,
            "SEARCH_TRAIN_PER_CLASS": SEARCH_TRAIN_PER_CLASS,
            "SEARCH_VAL_PER_CLASS": SEARCH_VAL_PER_CLASS,
        },
    }
    np.savez_compressed(masks_path, **arrays)
    save_json(metadata, meta_path)
    print(f"✅ HPO split masks saved: {masks_path}")
    print(f"✅ HPO split metadata saved: {meta_path}")


def load_hpo_split_cache(masks_path, meta_path):
    if not os.path.exists(masks_path) or not os.path.exists(meta_path):
        raise FileNotFoundError("Missing HPO split cache.")
    with open(meta_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    if int(metadata["rows"]) != int(rows) or int(metadata["cols"]) != int(cols):
        raise RuntimeError("HPO split cache shape mismatch.")
    data = np.load(masks_path, allow_pickle=False)
    splits = []
    for split_meta in metadata["splits"]:
        i = int(split_meta["index"])
        prefix = f"hpo_{i}"
        splits.append({
            "index": i,
            "name": split_meta["name"],
            "short": split_meta["short"],
            "position": split_meta["position"],
            "seed": split_meta.get("seed", 4000 + i),
            "rect": split_meta.get("rect", None),
            "train_mask": data[f"{prefix}_train_mask"].astype(bool),
            "val_mask": data[f"{prefix}_val_mask"].astype(bool),
            "val_buffer_mask": data[f"{prefix}_val_buffer_mask"].astype(bool),
            "train_coords": data[f"{prefix}_train_coords"].astype(np.int32),
            "val_coords": data[f"{prefix}_val_coords"].astype(np.int32),
        })
    return splits, metadata


def export_hpo_split_counts(global_hpo_splits, labels):
    rows_out = []
    for split in global_hpo_splits:
        rows_out.append(counts_record_from_mask(split["name"], "hpo_train", split["train_mask"], labels))
        rows_out.append(counts_record_from_mask(split["name"], "hpo_validation", split["val_mask"], labels))
        rows_out.append(counts_record_from_mask(split["name"], "hpo_validation_buffer", split["val_buffer_mask"], labels))
    df = pd.DataFrame(rows_out)
    path = os.path.join(hpo_dir, "hpo_spatial_split_class_counts.csv")
    cache_path = os.path.join(split_cache_dir, f"{HPO_SPLIT_CACHE_NAME}_class_counts.csv")
    df.to_csv(path, index=False)
    df.to_csv(cache_path, index=False)
    print(f"✅ HPO split class-count audit saved: {path}")
    print(df)
    return df



## 8. Hyperparameter optimization, training and model evaluation


In [ ]:
# ============================================================
# HPO + FINAL RUN LOGIC
# ============================================================
def arch_enabled_list():
    return [name for name in ["CNN", "ViT", "UNetPP", "ConvNeXt"] if ARCHITECTURE_ENABLED.get(name, False)]


def architecture_subdirs_for_arch(arch_name):
    """Return and create the complete per-architecture output tree."""
    base = os.path.join(out_dir, "architectures", str(arch_name))
    paths = {
        "base": base,
        "models": os.path.join(base, "models"),
        "figures": os.path.join(base, "figures"),
        "reports": os.path.join(base, "reports"),
        "hpo": os.path.join(base, "hpo"),
        "hpo_models": os.path.join(base, "hpo", "checkpoints"),
        "hpo_figures": os.path.join(base, "hpo", "figures"),
        "hpo_reports": os.path.join(base, "hpo", "reports"),
    }
    for _path in paths.values():
        os.makedirs(_path, exist_ok=True)
    return paths


def best_config_path_for_arch(arch_name):
    return os.path.join(architecture_subdirs_for_arch(arch_name)["hpo_reports"], f"best_global_config_{arch_name}.json")


def hpo_results_path_for_arch(arch_name):
    return os.path.join(architecture_subdirs_for_arch(arch_name)["hpo_reports"], f"global_hpo_results_{arch_name}.csv")


def hpo_split_scores_path_for_arch(arch_name):
    return os.path.join(architecture_subdirs_for_arch(arch_name)["hpo_reports"], f"global_hpo_split_scores_{arch_name}.csv")


def final_summary_path_for_arch(arch_name):
    return os.path.join(architecture_subdirs_for_arch(arch_name)["reports"], f"spatial_cv_fixed_config_test_summary_{arch_name}.csv")


def sanitize_cfg(cfg):
    keys = ["trial", "patch_size", "filters1", "filters2", "dense_units", "dropout", "lr", "batch_size", "epochs"]
    return {k: to_native(cfg[k]) for k in keys if k in cfg}


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_selected_config_for_arch(arch_name, stack, global_hpo_splits):
    arch_paths = architecture_subdirs_for_arch(arch_name)
    best_cfg_path = best_config_path_for_arch(arch_name)
    hpo_csv_path = hpo_results_path_for_arch(arch_name)
    hpo_split_scores_csv = hpo_split_scores_path_for_arch(arch_name)
    hpo_log_path = os.path.join(arch_paths["hpo_reports"], f"global_hpo_live_log_{arch_name}.txt")

    if SKIP_HPO_IF_BEST_CONFIG_EXISTS and os.path.exists(best_cfg_path):
        print(f"[{arch_name}] Existing best HPO config found. Loading: {best_cfg_path}")
        return sanitize_cfg(load_json(best_cfg_path))

    if not RUN_HPO_BY_ARCH.get(arch_name, False):
        if os.path.exists(best_cfg_path):
            print(f"[{arch_name}] HPO disabled; loading existing best config.")
            return sanitize_cfg(load_json(best_cfg_path))
        print(f"[{arch_name}] HPO disabled and no best config exists. Using manual fallback config.")
        fallback = dict(MANUAL_FALLBACK_CONFIG_BY_ARCH[arch_name])
        fallback["selected_without_hpo"] = True
        save_json(fallback, best_cfg_path)
        return sanitize_cfg(fallback)

    stage(f"HPO FOR ARCHITECTURE: {arch_name}")
    write_text(hpo_log_path, f"GLOBAL HPO LIVE LOG FOR {arch_name}\n")
    append_text(hpo_log_path, f"HPO_EPOCH_CAP={HPO_EPOCH_CAP} | HPO_USE_TRIAL_EPOCHS={HPO_USE_TRIAL_EPOCHS} | EARLY_STOPPING_PATIENCE_HPO={EARLY_STOPPING_PATIENCE_HPO} | RESTORE_BEST_WEIGHTS=True\n\n")
    save_json({
        "architecture": arch_name,
        "hpo_epoch_cap": HPO_EPOCH_CAP,
        "hpo_use_trial_epochs": HPO_USE_TRIAL_EPOCHS,
        "early_stopping_patience_hpo": EARLY_STOPPING_PATIENCE_HPO,
        "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
        "reduce_lr_patience_hpo": REDUCE_LR_PATIENCE_HPO,
        "param_configs": param_configs,
        "search_train_per_class": SEARCH_TRAIN_PER_CLASS,
        "search_val_per_class": SEARCH_VAL_PER_CLASS,
    }, os.path.join(arch_paths["hpo_reports"], f"hpo_run_config_{arch_name}.json"))

    existing_df = None
    completed_trials = set()
    hpo_results = []
    best_global_result = None
    best_global_mean_f1 = -1.0

    if os.path.exists(hpo_csv_path):
        try:
            existing_df = pd.read_csv(hpo_csv_path)
            if "trial" in existing_df.columns:
                completed_trials = set(existing_df["trial"].astype(int).tolist())
                hpo_results = existing_df.to_dict("records")
                if "mean_global_hpo_macro_f1" in existing_df.columns and len(existing_df) > 0:
                    best_row = existing_df.sort_values("mean_global_hpo_macro_f1", ascending=False).iloc[0].to_dict()
                    best_global_result = best_row
                    best_global_mean_f1 = float(best_row["mean_global_hpo_macro_f1"])
                print(f"[{arch_name}] Resuming HPO: found completed trials {sorted(completed_trials)}")
        except Exception as e:
            print(f"[{arch_name}] Could not resume existing HPO CSV; starting fresh. Reason: {e}")
            completed_trials = set()
            hpo_results = []

    for cfg in param_configs:
        trial_number = int(cfg["trial"])
        if trial_number in completed_trials:
            print(f"[{arch_name}] Skipping already completed HPO trial {trial_number}")
            continue

        effective_hpo_epochs = min(int(cfg["epochs"]), int(HPO_EPOCH_CAP)) if HPO_USE_TRIAL_EPOCHS else int(HPO_EPOCH_CAP)
        print(f"\n{arch_name} HPO TRIAL {trial_number}/{N_PARAM_TRIALS}: {sanitize_cfg(cfg)} | effective_hpo_epochs={effective_hpo_epochs}")
        append_text(hpo_log_path, f"START TRIAL {trial_number} | config={json.dumps(to_native(sanitize_cfg(cfg)))} | effective_hpo_epochs={effective_hpo_epochs}\n")
        trial_t0 = time.time()
        split_scores = []

        for split in global_hpo_splits:
            train_coords = np.asarray(split["train_coords"], dtype=np.int32)
            val_coords = np.asarray(split["val_coords"], dtype=np.int32)
            if len(train_coords) == 0 or len(val_coords) == 0:
                continue

            split_t0 = time.time()
            split_short = str(split.get("short", split.get("name", "HPO")))
            band_mean, band_std = compute_band_stats(stack, split["train_mask"], seed=GLOBAL_SEED + trial_number)
            fit_batch_size = int(HPO_BATCH_SIZE_OVERRIDE if A100_FAST_MODE else cfg["batch_size"])

            X_train = y_train = X_val = y_val_idx = None
            train_data = val_data = None
            if USE_PREEXTRACTED_PATCH_ARRAYS:
                X_train, y_train = extract_patch_array(
                    stack, ref_aligned, train_coords, cfg["patch_size"], band_mean, band_std,
                    dtype=PATCH_ARRAY_DTYPE,
                    progress_name=f"{arch_name} HPO T{trial_number} {split['short']} TRAIN",
                )
                X_val, y_val_idx = extract_patch_array(
                    stack, ref_aligned, val_coords, cfg["patch_size"], band_mean, band_std,
                    dtype=PATCH_ARRAY_DTYPE,
                    progress_name=f"{arch_name} HPO T{trial_number} {split['short']} VAL",
                )
            else:
                train_data = PatchSequence(stack, ref_aligned, train_coords, cfg["patch_size"], band_mean, band_std, batch_size=fit_batch_size, shuffle=True)
                val_data = PatchSequence(stack, ref_aligned, val_coords, cfg["patch_size"], band_mean, band_std, batch_size=fit_batch_size, shuffle=False)

            clear_mem(0.05)
            tf.random.set_seed(GLOBAL_SEED + 10000 + trial_number)
            model = ARCHITECTURE_BUILDERS[arch_name](
                cfg["patch_size"], INPUT_CHANNELS, cfg["filters1"], cfg["filters2"], cfg["dense_units"], cfg["dropout"], cfg["lr"]
            )

            hpo_tag = f"{arch_name}_HPO_T{trial_number}_{split_short}"
            hpo_model_path = os.path.join(arch_paths["hpo_models"], f"best_{hpo_tag}.keras")
            hpo_history_csv_path = os.path.join(arch_paths["hpo_reports"], f"history_{hpo_tag}.csv")
            hpo_summary_path = os.path.join(arch_paths["hpo_reports"], f"model_summary_{hpo_tag}.txt")

            if SAVE_MODEL_SUMMARIES:
                try:
                    summary_lines = []
                    model.summary(print_fn=lambda x: summary_lines.append(x))
                    write_text(hpo_summary_path, "\n".join(summary_lines))
                except Exception as e:
                    print(f"[{hpo_tag}] Could not save model summary:", e)

            es = callbacks.EarlyStopping(
                monitor="val_loss",
                patience=EARLY_STOPPING_PATIENCE_HPO,
                restore_best_weights=True,
                min_delta=EARLY_STOPPING_MIN_DELTA,
                verbose=0,
            )
            reduce_lr = callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.5,
                patience=REDUCE_LR_PATIENCE_HPO,
                min_lr=1e-6,
                verbose=1,
            )
            cb = [
                es,
                reduce_lr,
                callbacks.CSVLogger(hpo_history_csv_path),
                ProgressEpochPrinter(f"{arch_name} HPO T{trial_number} {split_short}"),
                MemoryGuardCallback(f"{arch_name} HPO T{trial_number}"),
            ]
            if SAVE_HPO_CHECKPOINTS:
                cb.insert(2, callbacks.ModelCheckpoint(
                    hpo_model_path,
                    monitor="val_loss",
                    save_best_only=True,
                    save_weights_only=False,
                    verbose=0,
                ))

            fit_t0 = time.time()
            if USE_PREEXTRACTED_PATCH_ARRAYS:
                history = model.fit(
                    X_train, y_train,
                    validation_data=(X_val, y_val_idx),
                    batch_size=fit_batch_size,
                    epochs=effective_hpo_epochs,
                    verbose=0,
                    callbacks=cb,
                )
                hpo_fit_time_sec = time.time() - fit_t0
                if SAVE_HPO_CHECKPOINTS and os.path.exists(hpo_model_path):
                    try:
                        model = tf.keras.models.load_model(hpo_model_path)
                    except Exception as e:
                        print(f"[{hpo_tag}] Could not reload HPO checkpoint; using restored current model:", e)
                prob = model.predict(X_val, batch_size=fit_batch_size, verbose=0)
                val_pred = np.array([index_to_label[int(k)] for k in np.argmax(prob, axis=1).astype(np.uint8)], dtype=np.uint8)
            else:
                history = model.fit(
                    train_data,
                    validation_data=val_data,
                    epochs=effective_hpo_epochs,
                    verbose=0,
                    callbacks=cb,
                )
                hpo_fit_time_sec = time.time() - fit_t0
                if SAVE_HPO_CHECKPOINTS and os.path.exists(hpo_model_path):
                    try:
                        model = tf.keras.models.load_model(hpo_model_path)
                    except Exception as e:
                        print(f"[{hpo_tag}] Could not reload HPO checkpoint; using restored current model:", e)
                val_pred = batch_patch_predict(model, stack, val_coords, cfg["patch_size"], band_mean, band_std, batch_size=VAL_INFER_BATCH_SIZE)

            hpo_epochs_ran = int(len(history.history.get("loss", [])))
            hpo_val_loss = history.history.get("val_loss", [])
            hpo_best_epoch = int(np.argmin(hpo_val_loss) + 1) if len(hpo_val_loss) else None
            hpo_best_val_loss = float(np.min(hpo_val_loss)) if len(hpo_val_loss) else None
            if SAVE_HPO_TRAINING_CURVES:
                try:
                    pd.DataFrame(history.history).to_csv(hpo_history_csv_path, index=False)
                    save_training_curve(history, os.path.join(arch_paths["hpo_figures"], f"training_curve_{hpo_tag}.png"), f"HPO training curve - {hpo_tag}")
                except Exception as e:
                    print(f"[{hpo_tag}] Could not save HPO history/curve:", e)

            y_val_true = ref_aligned[val_coords[:, 0], val_coords[:, 1]]
            score = {
                "architecture": arch_name,
                "trial": trial_number,
                "split": split["name"],
                "effective_hpo_epochs": int(effective_hpo_epochs),
                "epochs_ran": int(hpo_epochs_ran),
                "best_epoch": hpo_best_epoch,
                "best_val_loss": hpo_best_val_loss,
                "hpo_fit_time_sec": float(hpo_fit_time_sec),
                "hpo_total_split_time_sec": float(time.time() - split_t0),
                "checkpoint_path": hpo_model_path if SAVE_HPO_CHECKPOINTS else "",
                "history_csv_path": hpo_history_csv_path,
                "val_macro_f1": float(f1_score(y_val_true, val_pred, average="macro", zero_division=0)),
                "val_balanced_acc": float(balanced_accuracy_score(y_val_true, val_pred)),
                "val_acc": float(accuracy_score(y_val_true, val_pred)),
            }
            split_scores.append(score)
            pd.DataFrame([score]).to_csv(hpo_split_scores_csv, mode="a", header=not os.path.exists(hpo_split_scores_csv), index=False)
            print(f"{split['name']} | VAL MACRO F1={score['val_macro_f1']:.4f} | BAL ACC={score['val_balanced_acc']:.4f} | epochs={hpo_epochs_ran} | best_epoch={hpo_best_epoch} | fit_time={hpo_fit_time_sec:.1f}s")

            del model, X_train, y_train, X_val, y_val_idx, train_data, val_data
            clear_mem(0.15)

        if not split_scores:
            print(f"[{arch_name}] Trial {trial_number} produced no split scores. Skipping.")
            continue

        mean_f1 = float(np.mean([s["val_macro_f1"] for s in split_scores]))
        std_f1 = float(np.std([s["val_macro_f1"] for s in split_scores]))
        mean_bal = float(np.mean([s["val_balanced_acc"] for s in split_scores]))
        mean_acc = float(np.mean([s["val_acc"] for s in split_scores]))

        result = {
            **sanitize_cfg(cfg),
            "effective_hpo_epochs": int(effective_hpo_epochs),
            "mean_hpo_epochs_ran": float(np.mean([s.get("epochs_ran", 0) for s in split_scores])),
            "mean_hpo_best_epoch": float(np.mean([s.get("best_epoch", 0) for s in split_scores if s.get("best_epoch") is not None])),
            "total_hpo_fit_time_sec": float(np.sum([s.get("hpo_fit_time_sec", 0.0) for s in split_scores])),
            "total_hpo_trial_time_sec": float(time.time() - trial_t0),
            "mean_global_hpo_macro_f1": mean_f1,
            "std_global_hpo_macro_f1": std_f1,
            "mean_global_hpo_balanced_acc": mean_bal,
            "mean_global_hpo_acc": mean_acc,
        }
        hpo_results.append(result)
        append_text(hpo_log_path, f"Trial {trial_number} | mean_macro_f1={mean_f1:.6f} | std={std_f1:.6f}\n")

        if mean_f1 > best_global_mean_f1:
            best_global_mean_f1 = mean_f1
            best_global_result = result
            save_json(best_global_result, best_cfg_path)

        pd.DataFrame(hpo_results).sort_values("mean_global_hpo_macro_f1", ascending=False).to_csv(hpo_csv_path, index=False)

    if best_global_result is None:
        raise RuntimeError(f"[{arch_name}] HPO failed: no best configuration was selected.")

    selected_cfg = sanitize_cfg(best_global_result)
    print(f"\n[{arch_name}] SELECTED GLOBAL CONFIGURATION:", selected_cfg)
    return selected_cfg


def train_evaluate_final_for_arch(arch_name, selected_cfg, stack, outer_folds):
    if not RUN_FINAL_BY_ARCH.get(arch_name, False):
        print(f"[{arch_name}] Final training/evaluation disabled.")
        return None

    final_summary_csv = final_summary_path_for_arch(arch_name)
    if SKIP_FINAL_IF_SUMMARY_EXISTS and os.path.exists(final_summary_csv):
        print(f"[{arch_name}] Existing final summary found. Skipping final run: {final_summary_csv}")
        return pd.read_csv(final_summary_csv)

    stage(f"FINAL 4-OUTER EVALUATION FOR ARCHITECTURE: {arch_name}")
    arch_paths = architecture_subdirs_for_arch(arch_name)
    arch_model_dir = arch_paths["models"]
    arch_report_dir = arch_paths["reports"]
    arch_fig_dir = arch_paths["figures"]
    save_json(sanitize_cfg(selected_cfg), os.path.join(arch_paths["base"], f"selected_config_{arch_name}.json"))

    outer_results_live_csv = os.path.join(arch_report_dir, f"outer_fixed_config_results_live_{arch_name}.csv")
    if os.path.exists(outer_results_live_csv):
        os.remove(outer_results_live_csv)

    eval_results = []
    patch_size = int(selected_cfg["patch_size"])
    fit_batch_size = int(FINAL_BATCH_SIZE_OVERRIDE if A100_FAST_MODE else selected_cfg["batch_size"])

    for fold_idx, fold in enumerate(outer_folds, 1):
        clear_mem(0.1)
        tf.random.set_seed(GLOBAL_SEED + 20000 + fold_idx)
        np.random.seed(GLOBAL_SEED + 20000 + fold_idx)
        random.seed(GLOBAL_SEED + 20000 + fold_idx)

        tag = f"{fold['name']}_{arch_name}_TRIAL{int(selected_cfg.get('trial', -1))}"
        stage(f"{arch_name} FINAL TRAINING {fold_idx}/{len(outer_folds)}: {tag}")
        fold_t0 = time.time()

        pred_path = os.path.join(arch_report_dir, f"predictions_{tag}.npz")
        model_path = os.path.join(arch_model_dir, f"best_{tag}.keras")
        report_path = os.path.join(arch_report_dir, f"report_{tag}.txt")
        history_csv_path = os.path.join(arch_report_dir, f"history_{tag}.csv")

        if SKIP_FOLD_IF_PREDICTIONS_EXIST and os.path.exists(pred_path):
            print(f"[{tag}] Existing predictions found. Reusing: {pred_path}")
            pred_npz = np.load(pred_path, allow_pickle=False)
            y_test = pred_npz["y_true"].astype(np.uint8)
            y_pred = pred_npz["y_pred"].astype(np.uint8)
            res = {"architecture": arch_name, "outer_fold": fold["name"], "trial": int(selected_cfg.get("trial", -1)), **metric_dict(y_test, y_pred, "test")}
            eval_results.append(res)
            pd.DataFrame([res]).to_csv(outer_results_live_csv, mode="a", header=not os.path.exists(outer_results_live_csv), index=False)
            continue

        train_coords = sample_balanced_coords(fold["final_train_mask"], ref_aligned, FINAL_TRAIN_PER_CLASS, seed=GLOBAL_SEED + 10 * fold_idx)
        val_coords = sample_balanced_coords(fold["final_val_mask"], ref_aligned, FINAL_VAL_PER_CLASS, seed=GLOBAL_SEED + 20 * fold_idx)

        if DO_FULL_TEST_INFERENCE:
            test_coords = get_coords(fold["test_mask"])
        else:
            test_coords = sample_balanced_coords(fold["test_mask"], ref_aligned, TEST_SAMPLE_PER_CLASS_IF_NOT_FULL, seed=GLOBAL_SEED + 30 * fold_idx)

        print("Sampled train counts:", class_counts_from_coords(train_coords, ref_aligned))
        print("Sampled val counts:  ", class_counts_from_coords(val_coords, ref_aligned))
        print("Test counts:         ", class_counts_from_coords(test_coords, ref_aligned))

        band_mean, band_std = compute_band_stats(stack, fold["final_train_mask"], seed=GLOBAL_SEED + fold_idx)
        save_json(
            {"fold": fold["name"], "band_names": band_names, "band_mean": band_mean.tolist(), "band_std": band_std.tolist()},
            os.path.join(arch_report_dir, f"normalization_stats_{fold['name']}_{arch_name}.json"),
        )

        X_train = y_train = X_val = y_val = None
        train_data = val_data = None
        if USE_PREEXTRACTED_PATCH_ARRAYS:
            X_train, y_train = extract_patch_array(
                stack, ref_aligned, train_coords, patch_size, band_mean, band_std,
                dtype=PATCH_ARRAY_DTYPE, progress_name=f"{tag} TRAIN",
            )
            X_val, y_val = extract_patch_array(
                stack, ref_aligned, val_coords, patch_size, band_mean, band_std,
                dtype=PATCH_ARRAY_DTYPE, progress_name=f"{tag} VAL",
            )
        else:
            train_data = PatchSequence(stack, ref_aligned, train_coords, patch_size, band_mean, band_std, batch_size=fit_batch_size, shuffle=True)
            val_data = PatchSequence(stack, ref_aligned, val_coords, patch_size, band_mean, band_std, batch_size=fit_batch_size, shuffle=False)

        model = ARCHITECTURE_BUILDERS[arch_name](
            patch_size,
            INPUT_CHANNELS,
            int(selected_cfg["filters1"]),
            int(selected_cfg["filters2"]),
            int(selected_cfg["dense_units"]),
            float(selected_cfg["dropout"]),
            float(selected_cfg["lr"]),
        )
        if fold_idx == 1:
            model.summary()
        if SAVE_MODEL_SUMMARIES:
            try:
                summary_lines = []
                model.summary(print_fn=lambda x: summary_lines.append(x))
                write_text(os.path.join(arch_report_dir, f"model_summary_{tag}.txt"), "\n".join(summary_lines))
            except Exception as e:
                print(f"[{tag}] Could not save model summary:", e)

        cb = [
            callbacks.EarlyStopping(
                monitor="val_loss",
                patience=EARLY_STOPPING_PATIENCE_FINAL,
                restore_best_weights=True,
                min_delta=EARLY_STOPPING_MIN_DELTA,
                verbose=1,
            ),
            callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.5,
                patience=REDUCE_LR_PATIENCE_FINAL,
                min_lr=1e-6,
                verbose=1,
            ),
            callbacks.ModelCheckpoint(
                model_path,
                monitor="val_loss",
                save_best_only=True,
                save_weights_only=False,
                verbose=1,
            ),
            callbacks.CSVLogger(history_csv_path),
            ProgressEpochPrinter(f"{tag} FINAL"),
            MemoryGuardCallback(f"{tag} FINAL"),
        ]

        train_t0 = time.time()
        if USE_PREEXTRACTED_PATCH_ARRAYS:
            history = model.fit(
                X_train,
                y_train,
                validation_data=(X_val, y_val),
                batch_size=fit_batch_size,
                epochs=int(FINAL_EPOCHS),
                shuffle=True,
                verbose=0,
                callbacks=cb,
            )
        else:
            history = model.fit(
                train_data,
                validation_data=val_data,
                epochs=int(FINAL_EPOCHS),
                verbose=0,
                callbacks=cb,
            )
        final_train_time_sec = time.time() - train_t0
        final_epochs_ran = int(len(history.history.get("loss", [])))
        final_val_loss_history = history.history.get("val_loss", [])
        final_best_epoch = int(np.argmin(final_val_loss_history) + 1) if len(final_val_loss_history) else None
        final_best_val_loss = float(np.min(final_val_loss_history)) if len(final_val_loss_history) else None

        try:
            if os.path.exists(model_path):
                model = tf.keras.models.load_model(model_path)
        except Exception as e:
            print("Could not reload checkpoint; using restored current model:", e)

        try:
            pd.DataFrame(history.history).to_csv(history_csv_path, index=False)
            save_training_curve(history, os.path.join(arch_fig_dir, f"training_curve_{tag}.png"), f"Training curve - {tag}")
            save_area_curve_panel(fold, history, os.path.join(arch_fig_dir, f"area_and_curve_{tag}.png"), f"{tag}: cached areas + training curve")
        except Exception as e:
            print(f"[{tag}] Could not save training figures/history:", e)

        # Validation metrics on sampled validation coords.
        y_val_true = ref_aligned[val_coords[:, 0], val_coords[:, 1]]
        val_pred_t0 = time.time()
        if USE_PREEXTRACTED_PATCH_ARRAYS:
            prob_val = model.predict(X_val, batch_size=fit_batch_size, verbose=0)
            y_val_pred = np.array([index_to_label[int(k)] for k in np.argmax(prob_val, axis=1).astype(np.uint8)], dtype=np.uint8)
        else:
            y_val_pred = batch_patch_predict(model, stack, val_coords, patch_size, band_mean, band_std, batch_size=VAL_INFER_BATCH_SIZE, progress_name=f"{tag} VAL")
        final_val_predict_time_sec = time.time() - val_pred_t0

        # Test metrics on full/sampled test coords.
        y_test = ref_aligned[test_coords[:, 0], test_coords[:, 1]].astype(np.uint8)
        test_pred_t0 = time.time()
        y_pred = batch_patch_predict(model, stack, test_coords, patch_size, band_mean, band_std, batch_size=INFER_BATCH_SIZE, progress_name=f"{tag} TEST")
        final_test_predict_time_sec = time.time() - test_pred_t0

        val_metrics = metric_dict(y_val_true, y_val_pred, "val")
        test_metrics = metric_dict(y_test, y_pred, "test")
        final_total_fold_time_sec = time.time() - fold_t0

        confusion_matrix_dataframe(y_test, y_pred).to_csv(os.path.join(arch_report_dir, f"confusion_matrix_test_{tag}.csv"))
        np.savez_compressed(pred_path, y_true=y_test.astype(np.uint8), y_pred=y_pred.astype(np.uint8), coords=np.asarray(test_coords, dtype=np.int32))

        report_text = (
            f"RUN: {tag}\n"
            f"ARCHITECTURE: {arch_name}\n"
            f"SELECTED CONFIG: {json.dumps(to_native(selected_cfg), indent=2)}\n"
            f"MODEL CHECKPOINT: {model_path}\n"
            f"HISTORY CSV: {history_csv_path}\n"
            f"FINAL_EPOCHS_CAP: {FINAL_EPOCHS}\n"
            f"EPOCHS_RAN: {final_epochs_ran}\n"
            f"BEST_EPOCH_BY_VAL_LOSS: {final_best_epoch}\n"
            f"BEST_VAL_LOSS: {final_best_val_loss}\n"
            f"TRAIN_TIME_SEC: {final_train_time_sec:.3f}\n"
            f"VAL_PREDICT_TIME_SEC: {final_val_predict_time_sec:.3f}\n"
            f"TEST_PREDICT_TIME_SEC: {final_test_predict_time_sec:.3f}\n"
            f"TOTAL_FOLD_TIME_SEC: {final_total_fold_time_sec:.3f}\n\n"
            f"VALIDATION METRICS:\n{json.dumps(to_native(val_metrics), indent=2)}\n\n"
            f"TEST METRICS:\n{json.dumps(to_native(test_metrics), indent=2)}\n\n"
            f"Classification Report:\n"
            f"{classification_report(y_test, y_pred, labels=list(valid_classes), target_names=labels_as_names(), zero_division=0)}\n"
        )
        write_text(report_path, report_text)

        res = {
            "architecture": arch_name,
            "outer_fold": fold["name"],
            "trial": int(selected_cfg.get("trial", -1)),
            "patch_size": int(selected_cfg["patch_size"]),
            "filters1": int(selected_cfg["filters1"]),
            "filters2": int(selected_cfg["filters2"]),
            "dense_units": int(selected_cfg["dense_units"]),
            "dropout": float(selected_cfg["dropout"]),
            "lr": float(selected_cfg["lr"]),
            "final_epochs_cap": int(FINAL_EPOCHS),
            "epochs_ran": int(final_epochs_ran),
            "best_epoch": final_best_epoch,
            "best_val_loss": final_best_val_loss,
            "train_time_sec": float(final_train_time_sec),
            "val_predict_time_sec": float(final_val_predict_time_sec),
            "test_predict_time_sec": float(final_test_predict_time_sec),
            "total_fold_time_sec": float(final_total_fold_time_sec),
            "model_checkpoint_path": model_path,
            "history_csv_path": history_csv_path,
            "predictions_path": pred_path,
            **val_metrics,
            **test_metrics,
        }
        eval_results.append(res)
        pd.DataFrame([res]).to_csv(outer_results_live_csv, mode="a", header=not os.path.exists(outer_results_live_csv), index=False)

        del model, X_train, y_train, X_val, y_val, train_data, val_data
        clear_mem(0.5)

    final_df = pd.DataFrame(eval_results)
    final_df.to_csv(final_summary_csv, index=False)

    aggregate = {
        "architecture": arch_name,
        "n_outer_folds": int(len(final_df)),
        "mean_epochs_ran": float(final_df["epochs_ran"].mean()) if "epochs_ran" in final_df.columns else None,
        "mean_best_epoch": float(final_df["best_epoch"].dropna().mean()) if "best_epoch" in final_df.columns and len(final_df["best_epoch"].dropna()) else None,
        "total_train_time_sec": float(final_df["train_time_sec"].sum()) if "train_time_sec" in final_df.columns else None,
        "total_test_predict_time_sec": float(final_df["test_predict_time_sec"].sum()) if "test_predict_time_sec" in final_df.columns else None,
        "total_wall_time_recorded_sec": float(final_df["total_fold_time_sec"].sum()) if "total_fold_time_sec" in final_df.columns else None,
        "mean_test_acc": float(final_df["test_acc"].mean()),
        "std_test_acc": float(final_df["test_acc"].std(ddof=0)),
        "mean_test_balanced_acc": float(final_df["test_balanced_acc"].mean()),
        "std_test_balanced_acc": float(final_df["test_balanced_acc"].std(ddof=0)),
        "mean_test_macro_f1": float(final_df["test_macro_f1"].mean()),
        "std_test_macro_f1": float(final_df["test_macro_f1"].std(ddof=0)),
        "mean_test_weighted_f1": float(final_df["test_weighted_f1"].mean()),
        "std_test_weighted_f1": float(final_df["test_weighted_f1"].std(ddof=0)),
        "mean_test_kappa": float(final_df["test_kappa"].mean()),
        "std_test_kappa": float(final_df["test_kappa"].std(ddof=0)),
    }
    agg_path = os.path.join(report_dir, f"aggregate_test_summary_{arch_name}.json")
    save_json(aggregate, agg_path)

    print(f"\n=== FINAL MEAN SCORES FOR {arch_name} ===")
    print("Mean Test Accuracy:      ", round(aggregate["mean_test_acc"], 4))
    print("Mean Test Balanced Acc:  ", round(aggregate["mean_test_balanced_acc"], 4))
    print("Mean Test Macro F1:      ", round(aggregate["mean_test_macro_f1"], 4))
    print("Mean Test Weighted F1:   ", round(aggregate["mean_test_weighted_f1"], 4))
    print("Mean Cohen Kappa:        ", round(aggregate["mean_test_kappa"], 4))

    return final_df


def build_model_comparison_summary():
    rows_out = []
    for arch_name in arch_enabled_list():
        summary_csv = final_summary_path_for_arch(arch_name)
        if not os.path.exists(summary_csv):
            continue
        df = pd.read_csv(summary_csv)
        if len(df) == 0:
            continue
        rows_out.append({
            "architecture": arch_name,
            "n_outer_folds": int(len(df)),
            "selected_trial": int(df["trial"].iloc[0]) if "trial" in df.columns else -1,
            "mean_epochs_ran": float(df["epochs_ran"].mean()) if "epochs_ran" in df.columns else None,
            "total_train_time_sec": float(df["train_time_sec"].sum()) if "train_time_sec" in df.columns else None,
            "total_fold_time_sec": float(df["total_fold_time_sec"].sum()) if "total_fold_time_sec" in df.columns else None,
            "mean_test_acc": float(df["test_acc"].mean()),
            "std_test_acc": float(df["test_acc"].std(ddof=0)),
            "mean_test_balanced_acc": float(df["test_balanced_acc"].mean()),
            "std_test_balanced_acc": float(df["test_balanced_acc"].std(ddof=0)),
            "mean_test_macro_f1": float(df["test_macro_f1"].mean()),
            "std_test_macro_f1": float(df["test_macro_f1"].std(ddof=0)),
            "mean_test_weighted_f1": float(df["test_weighted_f1"].mean()),
            "std_test_weighted_f1": float(df["test_weighted_f1"].std(ddof=0)),
            "mean_test_kappa": float(df["test_kappa"].mean()),
            "std_test_kappa": float(df["test_kappa"].std(ddof=0)),
        })
    comp = pd.DataFrame(rows_out)
    if len(comp) > 0:
        comp = comp.sort_values("mean_test_macro_f1", ascending=False)
    path = os.path.join(report_dir, "architecture_comparison_summary.csv")
    comp.to_csv(path, index=False)
    print(f"✅ Architecture comparison summary saved: {path}")
    print(comp)
    return comp



## 9. Run the experiment and save the reports


In [ ]:
# ============================================================
# MAIN PIPELINE
# ============================================================
stage("1/12 SPLIT CACHE DECISION")
cache_exists = os.path.exists(SPLIT_MASKS_PATH) and os.path.exists(SPLIT_META_PATH)

if not BUILD_OR_LOAD_SPLIT:
    raise RuntimeError("BUILD_OR_LOAD_SPLIT=False is not supported for this full pipeline; outer_folds must exist.")

if cache_exists and not OVERWRITE_EXISTING_SPLIT:
    print("Existing outer split cache found. Loading it instead of regenerating.")
    print("To rebuild, set OVERWRITE_EXISTING_SPLIT = True.")
    common_valid_centers, outer_folds, global_hpo_safe_mask_loaded, split_metadata = load_split_cache(
        labels=ref_aligned,
        masks_path=SPLIT_MASKS_PATH,
        meta_path=SPLIT_META_PATH,
    )
else:
    if cache_exists and OVERWRITE_EXISTING_SPLIT:
        print("Existing split cache found, but OVERWRITE_EXISTING_SPLIT=True. Regenerating.")
    common_valid_centers, outer_folds, global_hpo_safe_mask_loaded = build_outer_folds(ref_aligned)
    save_split_cache(
        common_valid_centers=common_valid_centers,
        outer_folds=outer_folds,
        global_hpo_safe_mask=global_hpo_safe_mask_loaded,
        masks_path=SPLIT_MASKS_PATH,
        meta_path=SPLIT_META_PATH,
    )
    split_metadata = None

union_outer_test_mask, union_outer_test_buffer_mask, global_hpo_safe_mask_recomputed = compute_union_outer_masks(outer_folds, common_valid_centers)
global_hpo_safe_mask = global_hpo_safe_mask_loaded if global_hpo_safe_mask_loaded is not None else global_hpo_safe_mask_recomputed

stage("2/12 EXPORT SPLIT AUDITS + VISUALIZATION")
save_json(split_config_dict(), os.path.join(out_dir, "split_generation_config.json"))
outer_split_counts_df = export_outer_split_class_counts(
    outer_folds=outer_folds,
    labels=ref_aligned,
    paths=[
        os.path.join(out_dir, "outer_spatial_split_class_counts.csv"),
        os.path.join(split_cache_dir, f"{SPLIT_CACHE_NAME}_class_counts.csv"),
    ],
    common_valid_centers=common_valid_centers,
)

main_fig_path = os.path.join(fig_dir, "random4outer_layouts.png")
cache_fig_path = os.path.join(split_cache_dir, f"{SPLIT_CACHE_NAME}_layouts.png")
save_outer_layout_figure(outer_folds, main_fig_path, common_valid_centers=common_valid_centers)
save_outer_layout_figure(outer_folds, cache_fig_path, common_valid_centers=common_valid_centers)
save_individual_fold_figures(outer_folds, fig_dir, common_valid_centers=common_valid_centers)
maybe_show_image(main_fig_path, title="Cached spatial split layouts")

stage("3/12 LOAD FULL STACK")
stack = load_full_stack_array()
INPUT_CHANNELS = int(stack.shape[-1])

stage("4/12 HPO SPLIT CACHE DECISION")
need_any_hpo = any(ARCHITECTURE_ENABLED.get(a, False) and RUN_HPO_BY_ARCH.get(a, False) for a in ARCHITECTURE_BUILDERS.keys())
# If every enabled architecture already has a best config, HPO subfolds are not needed.
if BUILD_HPO_SPLITS_ONLY_WHEN_NEEDED:
    missing_best = [a for a in arch_enabled_list() if RUN_HPO_BY_ARCH.get(a, False) and not os.path.exists(best_config_path_for_arch(a))]
    need_hpo_splits = len(missing_best) > 0
else:
    need_hpo_splits = need_any_hpo

global_hpo_splits = []
if need_hpo_splits:
    hpo_cache_exists = os.path.exists(HPO_SPLIT_MASKS_PATH) and os.path.exists(HPO_SPLIT_META_PATH)
    if hpo_cache_exists and not REBUILD_HPO_SPLITS:
        print("Existing HPO split cache found. Loading it.")
        global_hpo_splits, hpo_split_metadata = load_hpo_split_cache(HPO_SPLIT_MASKS_PATH, HPO_SPLIT_META_PATH)
    else:
        if hpo_cache_exists and REBUILD_HPO_SPLITS:
            print("Existing HPO split cache found, but REBUILD_HPO_SPLITS=True. Regenerating.")
        global_hpo_splits = build_all_spatial_global_hpo_splits(
            global_hpo_safe_mask=global_hpo_safe_mask,
            union_outer_test_mask=union_outer_test_mask,
            union_outer_test_buffer_mask=union_outer_test_buffer_mask,
        )
        save_hpo_split_cache(global_hpo_splits, HPO_SPLIT_MASKS_PATH, HPO_SPLIT_META_PATH)
    export_hpo_split_counts(global_hpo_splits, ref_aligned)
else:
    print("No HPO subfolds needed: HPO disabled or best configs already exist for enabled architectures.")

stage("5/12 SAVE RUN CONFIG")
run_config = {
    "architectures_enabled": ARCHITECTURE_ENABLED,
    "run_hpo_by_arch": RUN_HPO_BY_ARCH,
    "run_final_by_arch": RUN_FINAL_BY_ARCH,
    "param_configs": param_configs,
    "split_cache_name": SPLIT_CACHE_NAME,
    "split_masks_path": SPLIT_MASKS_PATH,
    "split_meta_path": SPLIT_META_PATH,
    "hpo_split_cache_name": HPO_SPLIT_CACHE_NAME,
    "hpo_split_masks_path": HPO_SPLIT_MASKS_PATH,
    "hpo_split_meta_path": HPO_SPLIT_META_PATH,
    "final_epochs": FINAL_EPOCHS,
    "hpo_epoch_cap": HPO_EPOCH_CAP,
    "final_train_per_class": FINAL_TRAIN_PER_CLASS,
    "final_val_per_class": FINAL_VAL_PER_CLASS,
    "search_train_per_class": SEARCH_TRAIN_PER_CLASS,
    "search_val_per_class": SEARCH_VAL_PER_CLASS,
    "do_full_test_inference": DO_FULL_TEST_INFERENCE,
    "save_hpo_checkpoints": SAVE_HPO_CHECKPOINTS,
    "save_hpo_training_curves": SAVE_HPO_TRAINING_CURVES,
    "save_model_summaries": SAVE_MODEL_SUMMARIES,
    "save_timing_logs": SAVE_TIMING_LOGS,
}
save_json(run_config, os.path.join(out_dir, "full_pipeline_run_config.json"))

stage("6/12 ARCHITECTURE LOOP")
selected_configs = {}
for arch_name in arch_enabled_list():
    if arch_name not in ARCHITECTURE_BUILDERS:
        print(f"Skipping unknown architecture: {arch_name}")
        continue
    try:
        stage(f"START ARCHITECTURE: {arch_name}")
        if RUN_HPO_BY_ARCH.get(arch_name, False) and (not (SKIP_HPO_IF_BEST_CONFIG_EXISTS and os.path.exists(best_config_path_for_arch(arch_name)))):
            if not global_hpo_splits:
                print(f"[{arch_name}] HPO requested but HPO splits were not loaded. Loading/building now.")
                hpo_cache_exists = os.path.exists(HPO_SPLIT_MASKS_PATH) and os.path.exists(HPO_SPLIT_META_PATH)
                if hpo_cache_exists and not REBUILD_HPO_SPLITS:
                    global_hpo_splits, _ = load_hpo_split_cache(HPO_SPLIT_MASKS_PATH, HPO_SPLIT_META_PATH)
                else:
                    global_hpo_splits = build_all_spatial_global_hpo_splits(
                        global_hpo_safe_mask=global_hpo_safe_mask,
                        union_outer_test_mask=union_outer_test_mask,
                        union_outer_test_buffer_mask=union_outer_test_buffer_mask,
                    )
                    save_hpo_split_cache(global_hpo_splits, HPO_SPLIT_MASKS_PATH, HPO_SPLIT_META_PATH)
                export_hpo_split_counts(global_hpo_splits, ref_aligned)

        selected_cfg = get_selected_config_for_arch(arch_name, stack, global_hpo_splits)
        selected_configs[arch_name] = selected_cfg
        train_evaluate_final_for_arch(arch_name, selected_cfg, stack, outer_folds)
        stage(f"COMPLETED ARCHITECTURE: {arch_name}")
    except Exception as e:
        print(f"\n❌ Architecture failed: {arch_name}")
        print("Error:", e)
        traceback.print_exc()
        if STOP_ON_MODEL_ERROR:
            raise
        else:
            continue

stage("11/12 FINAL ARCHITECTURE COMPARISON")
save_json(selected_configs, os.path.join(report_dir, "selected_configs_by_architecture.json"))
comparison_df = build_model_comparison_summary()

stage("12/12 DONE")
print("\n🎉 Full pipeline completed or skipped completed parts according to your flags.")
print("Outer split used for ALL architectures:")
print("  Masks:   ", SPLIT_MASKS_PATH)
print("  Metadata:", SPLIT_META_PATH)
print("Outputs:", out_dir)
print("Comparison summary:", os.path.join(report_dir, "architecture_comparison_summary.csv"))


## Optional: Package results generated in this Colab session
The commands below assume an experiment has completed and the expected output directory exists. They may generate a large ZIP file.


In [ ]:
!zip -r kerkini_outputs.zip /content/kerkini_optionB_all_models_hpo_final_v1

In [ ]:
from google.colab import files
files.download('kerkini_outputs.zip')